# Notebook 02 - Justifikasi Pemilihan Nilai k pada Algoritma KNN

**Dokumen revisi skripsi | DiaPredict - Prediksi Diabetes (Random Forest, KNN, SVM)**

---

## 1. Masalah yang Diangkat Penguji

Pada pengujian versi sebelumnya (notebook V2), nilai `n_neighbors` (nilai *k*) pada algoritma
K-Nearest Neighbors **tidak dipilih melalui analisis eksplisit**, melainkan diserahkan
sepenuhnya kepada `RandomizedSearchCV` atas tujuh kandidat saja, yaitu
`k = [3, 5, 7, 9, 11, 15, 21]`. Proses pencarian tersebut menghasilkan konfigurasi terbaik:

| Hyperparameter | Nilai hasil tuning V2 |
|---|---|
| `n_neighbors` | **21** |
| `weights` | `uniform` |
| `metric` | `euclidean` |
| `leaf_size` | `20` |
| Recall CV (5-fold) | 0.8803 |
| Gap train - validation (recall) | 0.0853 |

Konsekuensinya muncul tiga kelemahan metodologis yang wajar dipersoalkan penguji:

1. **Nilai k terpilih adalah nilai terbesar pada daftar kandidat.** Ketika optimum jatuh di
   ujung ruang pencarian, tidak ada bukti bahwa nilai itu benar-benar optimum - bisa jadi
   nilai k yang lebih besar (23, 31, 41, ...) justru lebih baik, tetapi tidak pernah diuji.
2. **Tidak ada analisis bias-variance.** Tidak ditunjukkan bagaimana performa berubah ketika
   k divariasikan, sehingga pemilihan k tampak sebagai keputusan "kotak hitam".
3. **Terdapat indikasi overfitting** (gap train-validation 0.0853) yang tidak dijelaskan
   penyebabnya maupun kaitannya dengan nilai k.

Notebook ini menjawab ketiga hal tersebut dengan eksperimen yang dapat direproduksi.

---

## 2. Landasan Teori: Trade-off Bias-Variance pada KNN

KNN adalah *non-parametric lazy learner*. Nilai k mengendalikan **kompleksitas model** secara
langsung, dan merupakan satu-satunya knob utama yang mengatur posisi model pada spektrum
bias-variance.

**k kecil (misalnya k = 1 sampai 5) - variance tinggi, bias rendah:**
- Prediksi hanya bergantung pada segelintir tetangga terdekat, sehingga satu titik data
  yang menyimpang (noise, outlier, kesalahan pencatatan HbA1c/glukosa) langsung mengubah
  keputusan klasifikasi.
- Batas keputusan (*decision boundary*) menjadi **bergerigi / tidak beraturan** karena
  mengikuti setiap detail lokal data latih.
- Akibatnya skor pada data latih sangat tinggi (pada k = 1 secara teori mendekati sempurna)
  tetapi skor validasi jauh lebih rendah. **Gap train-validation yang besar inilah tanda
  overfitting.**
- Secara formal, error prediksi KNN dapat diuraikan sebagai
  `Error = Bias^2 + Variance + Noise`, dengan komponen variance berbanding terbalik terhadap
  k (kira-kira proporsional terhadap `sigma^2 / k`).

**k besar (misalnya k = 41 sampai 51 atau lebih) - bias tinggi, variance rendah:**
- Prediksi merupakan rata-rata (voting) dari banyak tetangga, termasuk tetangga yang secara
  jarak sudah tidak lagi relevan secara medis.
- Batas keputusan menjadi **terlalu halus** dan cenderung mendekati aturan mayoritas global,
  sehingga struktur lokal data hilang.
- Model gagal menangkap pola sesungguhnya - inilah **underfitting**. Pada kasus ekstrem
  `k = n`, model selalu memprediksi kelas mayoritas.

**Nilai k yang baik** berada pada wilayah tengah: cukup besar untuk meredam noise (variance
turun, gap train-validation mengecil) namun belum cukup besar untuk mengaburkan struktur
lokal (bias belum naik tajam). Wilayah ini akan dibuktikan secara empiris pada Eksperimen 1
dan dipilih secara formal pada Eksperimen 2 memakai **aturan one-standard-error**.

---

## 3. Klarifikasi Penting: k = 20 atau k = 21?

Dalam catatan revisi, penguji menulis nilai **k = 20**. Perlu diklarifikasi bahwa nilai yang
benar-benar dihasilkan oleh proses tuning notebook V2 adalah **k = 21**, bukan 20. Perbedaan
ini bukan sekadar salah ketik, melainkan memiliki alasan metodologis:

- Seluruh kandidat k pada penelitian ini sengaja dibatasi pada **bilangan ganjil**. Pada
  klasifikasi biner (diabetes / tidak diabetes), k ganjil menjamin **tidak pernah terjadi
  seri (tie)** pada mekanisme *majority voting*. Bila k genap seperti 20, sangat mungkin
  terjadi 10 tetangga kelas positif berbanding 10 tetangga kelas negatif, dan keputusan
  akhirnya bergantung pada mekanisme *tie-breaking* internal scikit-learn (memilih label
  dengan indeks terkecil), yang bersifat sewenang-wenang dan tidak dapat dipertanggung-
  jawabkan secara klinis.
- Karena itu, seluruh sweep pada notebook ini pun hanya menguji **nilai k ganjil**
  (1, 3, 5, ..., 51). Nilai k = 20 tidak pernah termasuk ruang pencarian.
- Kesimpulannya: penyebutan "k = 20" pada catatan penguji merujuk pada model KNN hasil
  tuning V2, yang nilai sebenarnya adalah **k = 21**. Notebook ini akan menguji apakah
  k = 21 memang layak dipertahankan, atau perlu diganti dengan nilai lain hasil sweep.

---

## 4. Rancangan Eksperimen

| # | Eksperimen | Tujuan | Luaran |
|---|---|---|---|
| 1 | Sweep k = 1..51 (26 nilai ganjil) | Memetakan kurva bias-variance | `tabel_sweep_k`, `knn_kurva_bias_variance`, `knn_metrik_vs_k` |
| 2 | Aturan one-standard-error | Justifikasi formal pemilihan k | `tabel_one_se_kandidat`, `knn_one_se_rule` |
| 3 | Uji signifikansi antar-k | Membuktikan perbedaan k bermakna atau tidak | `tabel_uji_antar_k`, `knn_uji_antar_k` |
| 4 | Grid 2 dimensi k x weights dan k x metric | Menjawab kenapa `uniform` dan `euclidean` | `tabel_grid_knn`, `knn_heatmap_weights`, `knn_heatmap_metric` |
| 5 | Heuristik sqrt(n) dan analisis sensitivitas | Membandingkan dengan aturan praktis, uji kestabilan | `tabel_heuristik_k`, `tabel_sensitivitas_k`, `knn_sensitivitas_k` |
| 6 | Verifikasi pada test set 20% | Konfirmasi pada data yang belum pernah dilihat | `tabel_verifikasi_k_testset`, `knn_verifikasi_testset` |

Seluruh eksperimen memakai pipeline anti-kebocoran data
(`StandardScaler` -> `SMOTE` -> `KNeighborsClassifier`) di dalam `imblearn.Pipeline`,
sehingga penskalaan dan oversampling hanya dihitung dari fold latih.

**Metrik utama = Recall kelas positif (diabetes)**, sesuai konteks skrining kesehatan:
kesalahan melewatkan penderita diabetes (*false negative*) jauh lebih berbahaya daripada
kesalahan memberi peringatan palsu (*false positive*).

In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

---

## CELL 7 - Konfigurasi Eksperimen

Beberapa keputusan desain yang perlu dicatat pada bagian metodologi skripsi:

1. **Split 80:20 dilakukan lebih dulu.** Seluruh eksperimen pemilihan k (sweep, one-SE rule,
   grid, sensitivitas) **hanya menggunakan data train (80%)**. Data test (20%) disimpan
   utuh dan baru dibuka pada Eksperimen 6 untuk verifikasi akhir. Dengan cara ini nilai k
   tidak pernah "mengintip" data test, sehingga hasil verifikasi tetap tidak bias.
2. **Ruang pencarian k = 1, 3, 5, ..., 51** (26 nilai ganjil). Batas atas 51 dipilih karena
   pada eksperimen pendahuluan kurva recall sudah jelas mendatar/menurun jauh sebelum titik
   tersebut, sehingga rentang ini cukup untuk memperlihatkan kedua ujung spektrum
   bias-variance. Hanya nilai ganjil yang diuji untuk menghindari seri pada voting biner.
3. **Validasi silang Stratified 5-Fold** dengan `shuffle=True` dan seed tetap, sama persis
   dengan skema notebook V2 agar angkanya dapat diperbandingkan langsung.
4. **`MODE_CEPAT`**: bila `True`, CV dijalankan pada subsample stratified 30.000 baris agar
   notebook selesai dalam hitungan menit di Colab. Untuk angka final skripsi, set
   `MODE_CEPAT = False` sehingga seluruh 76.916 baris data train dipakai.

In [ ]:
# ============================================================
# CELL 7: Konfigurasi Eksperimen Pemilihan k
# ============================================================

MODE_CEPAT  = True      # True  -> subsample stratified (uji cepat di Colab)
                        # False -> data train penuh (untuk angka final skripsi)
N_SUBSAMPLE = 30000     # ukuran subsample stratified saat MODE_CEPAT = True
N_FOLD      = 5         # jumlah fold CV utama (sama dengan notebook V2)
N_FOLD_RINGAN = 3       # fold untuk eksperimen berat (grid & sensitivitas)
N_REPEAT_UJI  = 3       # jumlah pengulangan CV untuk uji signifikansi antar-k
RASIO_TEST  = 0.2       # rasio split hasil justifikasi notebook 01

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X): return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

# --- Ruang pencarian nilai k ---------------------------------------------
# 26 nilai ganjil: 1, 3, 5, ..., 51. Hanya ganjil supaya majority voting pada
# klasifikasi biner tidak pernah menghasilkan seri (tie).
DAFTAR_K = list(range(1, 52, 2))

# Subset k untuk eksperimen berat (grid 2 dimensi & analisis sensitivitas)
DAFTAR_K_GRID = [1, 5, 9, 15, 21, 31, 41, 51]
DAFTAR_K_SENS = [1, 3, 5, 9, 15, 21, 31, 41, 51]

# --- Baseline hasil notebook V2 (yang ditulis penguji sebagai "k = 20") ---
K_BASELINE_V2   = PARAM_KNN_V2['n_neighbors']   # = 21
RECALL_CV_V2    = 0.8803
GAP_TRAIN_VAL_V2 = 0.0853

# --- Skema validasi silang ------------------------------------------------
CV        = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=RANDOM_STATE)
CV_RINGAN = StratifiedKFold(n_splits=N_FOLD_RINGAN, shuffle=True, random_state=RANDOM_STATE)

# --- Split 80:20 DULU: data test dikunci, tidak dipakai untuk memilih k ---
X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_all, y_all, test_size=RASIO_TEST, stratify=y_all, random_state=RANDOM_STATE
)

if MODE_CEPAT:
    X_cv, y_cv = ambil_subsample(X_tr_full, y_tr_full, N_SUBSAMPLE)
else:
    X_cv, y_cv = X_tr_full, y_tr_full

N_TRAIN_PENUH = len(X_tr_full)

# --- Fungsi aturan one-standard-error (dipakai CELL 8 dan CELL 11) --------
def pilih_k_one_se(tabel, kol_mean='val_recall_mean', kol_std='val_recall_std',
                   n_fold=N_FOLD):
    """Aturan one-standard-error (Hastie, Tibshirani & Friedman, 2009).

    Langkah:
      1) cari skor CV terbaik dan standard error-nya (SE = std / sqrt(n_fold));
      2) tetapkan ambang = skor_terbaik - 1 SE;
      3) di antara semua k yang skornya >= ambang, pilih MODEL PALING SEDERHANA.
         Pada KNN, model paling sederhana = k TERBESAR, karena semakin besar k
         semakin halus batas keputusan dan semakin rendah variance model.
    """
    t = tabel.copy().reset_index(drop=True)
    t['se'] = t[kol_std] / np.sqrt(n_fold)
    idx_best  = int(t[kol_mean].idxmax())
    skor_best = float(t.loc[idx_best, kol_mean])
    se_best   = float(t.loc[idx_best, 'se'])
    ambang    = skor_best - se_best
    kandidat  = t[t[kol_mean] >= ambang].copy()
    return {
        'k_terbaik'   : int(t.loc[idx_best, 'k']),
        'skor_terbaik': skor_best,
        'se_terbaik'  : se_best,
        'ambang_1se'  : ambang,
        'k_terpilih'  : int(kandidat['k'].max()),
        'kandidat'    : kandidat,
    }

garis('KONFIGURASI EKSPERIMEN PEMILIHAN k')
print(f'MODE_CEPAT             : {MODE_CEPAT}')
print(f'Total data bersih      : {len(X_all):,} baris')
print(f'Data train (80%)       : {N_TRAIN_PENUH:,} baris  <- dipakai memilih k')
print(f'Data test  (20%)       : {len(X_te):,} baris  <- DIKUNCI sampai Eksperimen 6')
print(f'Data untuk CV          : {len(X_cv):,} baris '
      f'({"subsample stratified" if MODE_CEPAT else "train penuh"})')
print(f'Positif pada data CV   : {int(y_cv.sum()):,} ({y_cv.mean()*100:.2f}%)')
print('')
print(f'Ruang pencarian k      : {DAFTAR_K}')
print(f'Jumlah nilai k diuji   : {len(DAFTAR_K)} (semua ganjil)')
print(f'Skema CV utama         : StratifiedKFold(n_splits={N_FOLD}, shuffle=True)')
print(f'Baseline notebook V2   : k={K_BASELINE_V2}, recall CV={RECALL_CV_V2:.4f}, '
      f'gap train-val={GAP_TRAIN_VAL_V2:.4f}')
garis()
print('CATATAN: k=20 (genap) TIDAK termasuk ruang pencarian. Nilai hasil tuning V2')
print('         yang sebenarnya adalah k=21. Lihat penjelasan pada markdown pembuka.')

---

# EKSPERIMEN 1 - Sweep Nilai k Secara Menyeluruh

Setiap nilai k pada `DAFTAR_K` dievaluasi dengan `cross_validate` 5-fold. Yang dicatat bukan
hanya skor validasi, tetapi juga **skor pada fold latih** (`return_train_score=True`).
Selisih keduanya (*gap* train - validation) adalah indikator kuantitatif overfitting:
gap besar berarti model menghafal data latih, gap kecil berarti model menggeneralisasi.

Empat metrik dicatat sekaligus:

- **Recall** - metrik utama (proporsi penderita diabetes yang berhasil terdeteksi);
- **Precision** - proporsi prediksi positif yang benar (mengukur beban *false alarm*);
- **F1-score** - rata-rata harmonik recall dan precision;
- **ROC-AUC** - kemampuan pemeringkatan model, tidak bergantung pada threshold.

In [ ]:
# ============================================================
# CELL 8: EKSPERIMEN 1 - Sweep k = 1..51 (26 nilai ganjil)
# ============================================================
METRIK_UJI = ['recall', 'precision', 'f1', 'roc_auc']

garis('EKSPERIMEN 1: SWEEP NILAI k')
print(f'Nilai k diuji      : {len(DAFTAR_K)} nilai -> {DAFTAR_K}')
print(f'Total fit model    : {len(DAFTAR_K) * N_FOLD} kali ({len(DAFTAR_K)} k x {N_FOLD} fold)')
print(f'Ukuran data CV     : {len(X_cv):,} baris')
print(f'Estimasi waktu     : sekitar {len(DAFTAR_K) * N_FOLD * 0.6 / 60:.1f}-'
      f'{len(DAFTAR_K) * N_FOLD * 2.5 / 60:.1f} menit pada Colab CPU')
print('')

baris_sweep = []
skor_fold_recall = {}          # k -> array recall per fold (untuk one-SE & uji statistik)
t_mulai = time.time()

for i, k in enumerate(DAFTAR_K, start=1):
    t0 = time.time()
    pipe = buat_pipeline_knn(n_neighbors=k)
    res = cross_validate(
        pipe, X_cv, y_cv, cv=CV,
        scoring=METRIK_UJI,
        return_train_score=True,
        n_jobs=1,
        error_score='raise',
    )
    durasi = time.time() - t0

    skor_fold_recall[k] = np.asarray(res['test_recall'], dtype=float)

    baris = {'k': int(k)}
    for m in METRIK_UJI:
        tr = np.asarray(res[f'train_{m}'], dtype=float)
        va = np.asarray(res[f'test_{m}'],  dtype=float)
        baris[f'train_{m}_mean'] = float(tr.mean())
        baris[f'train_{m}_std']  = float(tr.std(ddof=1))
        baris[f'val_{m}_mean']   = float(va.mean())
        baris[f'val_{m}_std']    = float(va.std(ddof=1))
        baris[f'gap_{m}']        = float(tr.mean() - va.mean())
    baris['waktu_fit_s']   = float(np.sum(res['fit_time']))
    baris['waktu_skor_s']  = float(np.sum(res['score_time']))
    baris['waktu_total_s'] = float(durasi)
    baris_sweep.append(baris)

    print(f'[{i:2d}/{len(DAFTAR_K)}] k={k:2d} | '
          f'recall val={baris["val_recall_mean"]:.4f} (+/-{baris["val_recall_std"]:.4f}) | '
          f'train={baris["train_recall_mean"]:.4f} | '
          f'gap={baris["gap_recall"]:+.4f} | '
          f'f1={baris["val_f1_mean"]:.4f} | auc={baris["val_roc_auc_mean"]:.4f} | '
          f'{durasi:.1f}s')

print('')
print(f'Total waktu Eksperimen 1 : {(time.time() - t_mulai)/60:.2f} menit')
print('')

tabel_sweep_k = pd.DataFrame(baris_sweep)
kolom_ringkas = ['k', 'train_recall_mean', 'val_recall_mean', 'val_recall_std',
                 'gap_recall', 'val_precision_mean', 'val_f1_mean',
                 'val_roc_auc_mean', 'waktu_total_s']
simpan_tabel(tabel_sweep_k, 'tabel_sweep_k', tampilkan=False)
display(tabel_sweep_k[kolom_ringkas].round(4))

# --- Ringkasan temuan -----------------------------------------------------
K_TERBAIK_CV  = int(tabel_sweep_k.loc[tabel_sweep_k['val_recall_mean'].idxmax(), 'k'])
K_TERBAIK_F1  = int(tabel_sweep_k.loc[tabel_sweep_k['val_f1_mean'].idxmax(), 'k'])
K_TERBAIK_AUC = int(tabel_sweep_k.loc[tabel_sweep_k['val_roc_auc_mean'].idxmax(), 'k'])
K_GAP_MIN     = int(tabel_sweep_k.loc[tabel_sweep_k['gap_recall'].abs().idxmin(), 'k'])

_one_se_awal = pilih_k_one_se(tabel_sweep_k)
K_TERPILIH   = int(_one_se_awal['k_terpilih'])   # difinalkan & dibedah di Eksperimen 2

b_k1  = tabel_sweep_k[tabel_sweep_k['k'] == 1].iloc[0]
b_bar = tabel_sweep_k[tabel_sweep_k['k'] == K_TERBAIK_CV].iloc[0]
b_v2  = tabel_sweep_k[tabel_sweep_k['k'] == K_BASELINE_V2].iloc[0]
b_max = tabel_sweep_k[tabel_sweep_k['k'] == max(DAFTAR_K)].iloc[0]

garis('RINGKASAN EKSPERIMEN 1')
print(f'k dengan recall CV tertinggi   : k={K_TERBAIK_CV} '
      f'(recall={b_bar["val_recall_mean"]:.4f})')
print(f'k dengan F1 tertinggi          : k={K_TERBAIK_F1}')
print(f'k dengan ROC-AUC tertinggi     : k={K_TERBAIK_AUC}')
print(f'k dengan gap train-val terkecil: k={K_GAP_MIN}')
print(f'k terpilih (aturan one-SE)     : k={K_TERPILIH}  <- dibuktikan di Eksperimen 2')
print('')
print('BUKTI BIAS-VARIANCE:')
print(f'  k=1  (variance tinggi) : train recall={b_k1["train_recall_mean"]:.4f} | '
      f'val recall={b_k1["val_recall_mean"]:.4f} | gap={b_k1["gap_recall"]:+.4f}')
print(f'  k={K_BASELINE_V2} (baseline V2)  : train recall={b_v2["train_recall_mean"]:.4f} | '
      f'val recall={b_v2["val_recall_mean"]:.4f} | gap={b_v2["gap_recall"]:+.4f}')
print(f'  k={max(DAFTAR_K)} (bias tinggi)  : train recall={b_max["train_recall_mean"]:.4f} | '
      f'val recall={b_max["val_recall_mean"]:.4f} | gap={b_max["gap_recall"]:+.4f}')
print('')
print(f'  Gap pada k=1 lebih besar {b_k1["gap_recall"] - b_max["gap_recall"]:+.4f} '
      f'dibanding k={max(DAFTAR_K)} -> konfirmasi bahwa k kecil memang overfit.')
print('')
print(f'Perbandingan dengan notebook V2 (k={K_BASELINE_V2}):')
print(f'  Recall CV V2 dilaporkan  : {RECALL_CV_V2:.4f}')
print(f'  Recall CV notebook ini   : {b_v2["val_recall_mean"]:.4f} '
      f'(selisih {b_v2["val_recall_mean"] - RECALL_CV_V2:+.4f})')
print(f'  Gap V2 dilaporkan        : {GAP_TRAIN_VAL_V2:.4f}')
print(f'  Gap notebook ini         : {b_v2["gap_recall"]:.4f}')
print('  (Selisih kecil wajar bila MODE_CEPAT=True karena memakai subsample.)')

In [ ]:
# ============================================================
# CELL 9: VISUALISASI 1 - Kurva Bias-Variance (train vs validation)
# ============================================================
ks   = tabel_sweep_k['k'].values
tr_m = tabel_sweep_k['train_recall_mean'].values
tr_s = tabel_sweep_k['train_recall_std'].values
va_m = tabel_sweep_k['val_recall_mean'].values
va_s = tabel_sweep_k['val_recall_std'].values
gap  = tabel_sweep_k['gap_recall'].values

WARNA_TRAIN = '#34495e'
WARNA_VALID = WARNA_MODEL['KNN']

# Batas zona: sepertiga awal = zona variance tinggi, sepertiga akhir = zona bias tinggi
BATAS_OVERFIT   = 7
BATAS_UNDERFIT  = 35

fig, axes = plt.subplots(2, 1, figsize=(13, 11), sharex=True,
                         gridspec_kw={'height_ratios': [2.1, 1]})

# ---------- Panel atas: train vs validation recall ----------
ax = axes[0]
ax.plot(ks, tr_m, marker='o', ms=5, lw=2.2, color=WARNA_TRAIN,
        label='Recall pada data LATIH (train)')
ax.fill_between(ks, tr_m - tr_s, tr_m + tr_s, color=WARNA_TRAIN, alpha=0.15)
ax.plot(ks, va_m, marker='s', ms=5, lw=2.6, color=WARNA_VALID,
        label='Recall pada data VALIDASI (5-fold CV)')
ax.fill_between(ks, va_m - va_s, va_m + va_s, color=WARNA_VALID, alpha=0.18,
                label='Pita +/- 1 standar deviasi antar-fold')

ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.4,
           label=f'k terpilih = {K_TERPILIH} (aturan one-SE)')
ax.axvline(K_TERBAIK_CV, color='#8e44ad', ls=':', lw=2.0,
           label=f'k recall CV tertinggi = {K_TERBAIK_CV}')
if K_BASELINE_V2 not in (K_TERPILIH, K_TERBAIK_CV):
    ax.axvline(K_BASELINE_V2, color='#16a085', ls='-.', lw=1.8,
               label=f'k baseline notebook V2 = {K_BASELINE_V2}')

y_lo, y_hi = ax.get_ylim()
rentang = y_hi - y_lo
ax.axvspan(0, BATAS_OVERFIT, color='#e74c3c', alpha=0.07, zorder=0)
ax.axvspan(BATAS_UNDERFIT, max(ks) + 1, color='#3498db', alpha=0.07, zorder=0)
ax.text(1.2, y_lo + 0.06 * rentang,
        'ZONA OVERFITTING\n(variance tinggi, bias rendah)\nbatas keputusan bergerigi,\ngap train-validasi lebar',
        fontsize=9.5, color='#a93226', va='bottom', ha='left', fontweight='bold')
ax.text(max(ks) - 0.5, y_lo + 0.06 * rentang,
        'ZONA UNDERFITTING\n(bias tinggi, variance rendah)\nbatas keputusan terlalu halus,\nstruktur lokal hilang',
        fontsize=9.5, color='#1f618d', va='bottom', ha='right', fontweight='bold')
ax.set_ylim(y_lo, y_hi)

ax.set_ylabel('Recall (kelas diabetes)')
ax.set_title('Kurva Bias-Variance KNN: Recall Train vs Validation terhadap Nilai k\n'
             f'(Stratified {N_FOLD}-Fold CV pada {len(X_cv):,} baris data train)',
             fontweight='bold')
ax.legend(loc='center right', fontsize=9.5, framealpha=0.93)

# ---------- Panel bawah: gap train - validation ----------
ax2 = axes[1]
warna_bar = [WARNA_MODEL['KNN'] if g > 0.05 else
             (WARNA_AKSEN if g > 0.02 else '#2ecc71') for g in gap]
ax2.bar(ks, gap, width=1.4, color=warna_bar, edgecolor='white', lw=0.6)
ax2.plot(ks, gap, color='#2c3e50', lw=1.4, alpha=0.65)
ax2.axhline(0, color='#2c3e50', lw=1.0)
ax2.axhline(GAP_TRAIN_VAL_V2, color='#16a085', ls='-.', lw=1.8,
            label=f'Gap notebook V2 pada k={K_BASELINE_V2} = {GAP_TRAIN_VAL_V2:.4f}')
ax2.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.2)

idx_k1 = int(np.where(ks == ks.min())[0][0])
ax2.annotate(f'gap terbesar\n{gap[idx_k1]:+.4f}',
             xy=(ks[idx_k1], gap[idx_k1]),
             xytext=(ks[idx_k1] + 6, gap[idx_k1]),
             fontsize=9.5, color='#a93226', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#a93226', lw=1.4))

ax2.set_xlabel('Nilai k (jumlah tetangga terdekat)')
ax2.set_ylabel('Gap = recall train - recall validasi')
ax2.set_title('Indikator Overfitting: Semakin Besar Gap, Semakin Kuat Model Menghafal Data Latih',
              fontweight='bold')
ax2.legend(loc='upper right', fontsize=9.5)
ax2.set_xticks(ks)
ax2.tick_params(axis='x', labelsize=8.5)

plt.tight_layout()
simpan_gambar('knn_kurva_bias_variance')
plt.show()

garis('INTERPRETASI VISUALISASI 1')
print(f'1. Pada k=1 recall train mencapai {tr_m[0]:.4f} sementara recall validasi hanya '
      f'{va_m[0]:.4f}.')
print(f'   Gap sebesar {gap[0]:+.4f} adalah bukti langsung overfitting: model praktis')
print('   menghafal setiap titik data latih namun gagal menggeneralisasi.')
print(f'2. Gap menyusut secara monoton seiring naiknya k, dari {gap[0]:+.4f} pada k=1')
print(f'   menjadi {gap[-1]:+.4f} pada k={ks[-1]} - persis seperti yang diprediksi teori')
print('   bahwa komponen variance berbanding terbalik terhadap k.')
print(f'3. Recall validasi mencapai puncak pada k={K_TERBAIK_CV} '
      f'({va_m.max():.4f}) lalu bergerak mendatar/menurun,')
print('   menandakan bias mulai mendominasi pada k yang lebih besar.')
print(f'4. Wilayah stabil (dataran) inilah yang menjadi dasar pemilihan k pada Eksperimen 2.')

In [ ]:
# ============================================================
# CELL 10: VISUALISASI 2 - Semua Metrik Validasi terhadap k
# ============================================================
WARNA_METRIK = {
    'recall'   : WARNA_MODEL['KNN'],
    'precision': WARNA_MODEL['Random Forest'],
    'f1'       : WARNA_MODEL['SVM (Linear)'],
    'roc_auc'  : '#8e44ad',
}
LABEL_METRIK = {
    'recall'   : 'Recall (metrik utama)',
    'precision': 'Precision',
    'f1'       : 'F1-score',
    'roc_auc'  : 'ROC-AUC',
}
MARKER_METRIK = {'recall': 's', 'precision': 'o', 'f1': '^', 'roc_auc': 'D'}

fig, ax = plt.subplots(figsize=(13, 7))

for m in METRIK_UJI:
    nilai = tabel_sweep_k[f'val_{m}_mean'].values
    std   = tabel_sweep_k[f'val_{m}_std'].values
    lw    = 2.8 if m == 'recall' else 1.9
    ax.plot(ks, nilai, marker=MARKER_METRIK[m], ms=5, lw=lw,
            color=WARNA_METRIK[m], label=LABEL_METRIK[m])
    ax.fill_between(ks, nilai - std, nilai + std, color=WARNA_METRIK[m], alpha=0.10)
    k_top = int(tabel_sweep_k.loc[tabel_sweep_k[f'val_{m}_mean'].idxmax(), 'k'])
    ax.scatter([k_top], [nilai.max()], s=130, facecolors='none',
               edgecolors=WARNA_METRIK[m], lw=2.2, zorder=5)

ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.4,
           label=f'k terpilih = {K_TERPILIH}')
if K_BASELINE_V2 != K_TERPILIH:
    ax.axvline(K_BASELINE_V2, color='#16a085', ls='-.', lw=1.8,
               label=f'k baseline V2 = {K_BASELINE_V2}')

ax.set_xlabel('Nilai k (jumlah tetangga terdekat)')
ax.set_ylabel('Skor validasi (rata-rata 5-fold CV)')
ax.set_title('Perbandingan Seluruh Metrik Validasi terhadap Nilai k pada KNN\n'
             '(lingkaran kosong = nilai puncak tiap metrik)', fontweight='bold')
ax.set_xticks(ks)
ax.tick_params(axis='x', labelsize=8.5)
ax.legend(loc='center right', fontsize=10, framealpha=0.93)

plt.tight_layout()
simpan_gambar('knn_metrik_vs_k')
plt.show()

garis('INTERPRETASI VISUALISASI 2')
for m in METRIK_UJI:
    kolom = f'val_{m}_mean'
    k_top = int(tabel_sweep_k.loc[tabel_sweep_k[kolom].idxmax(), 'k'])
    v_top = float(tabel_sweep_k[kolom].max())
    v_k1  = float(tabel_sweep_k.loc[tabel_sweep_k['k'] == 1, kolom].iloc[0])
    print(f'{LABEL_METRIK[m]:<24}: puncak pada k={k_top:2d} (nilai={v_top:.4f}), '
          f'pada k=1 hanya {v_k1:.4f}')
print('')
print('Terlihat tarik-menarik antar metrik: recall cenderung naik seiring k karena SMOTE')
print('memperluas wilayah kelas positif, sementara precision menurun karena semakin banyak')
print('kasus sehat yang ikut tertandai. F1-score menjadi penyeimbang, dan ROC-AUC')
print('menunjukkan bahwa kualitas pemeringkatan model relatif stabil pada k menengah-besar.')
print('Karena konteks penelitian adalah SKRINING KESEHATAN, recall dijadikan metrik utama.')

---

# EKSPERIMEN 2 - Aturan One-Standard-Error (Justifikasi Formal)

Memilih k yang skor CV-nya paling tinggi terdengar masuk akal, tetapi secara statistik
**berisiko**. Skor CV adalah estimasi dengan ketidakpastian; perbedaan 0,001 antara dua nilai k
sering kali hanya *noise* dari pembagian fold, bukan keunggulan nyata. Memaksakan pilihan pada
nilai puncak berarti ikut mengoptimalkan noise tersebut (*selection overfitting*).

**Aturan one-standard-error** (Hastie, Tibshirani & Friedman, *The Elements of Statistical
Learning*, 2009) mengatasi hal ini dengan tiga langkah:

1. Hitung skor CV terbaik dan standard error-nya: `SE = std_antar_fold / sqrt(n_fold)`.
2. Tetapkan ambang toleransi: `ambang = skor_terbaik - 1 x SE`.
3. Di antara semua kandidat yang skornya masih di atas ambang, pilih **model paling
   sederhana**.

**Mengapa "paling sederhana" pada KNN berarti k TERBESAR?** Karena kompleksitas efektif KNN
berbanding terbalik dengan k. Dengan n data latih, jumlah derajat kebebasan efektif KNN kira-kira
`n / k`: pada k = 1 model punya kompleksitas maksimum (setiap titik menjadi aturannya sendiri),
sedangkan pada k besar banyak titik berbagi satu keputusan sehingga batas keputusan menjadi
halus dan stabil. Jadi di antara nilai-nilai k yang performanya **secara statistik tidak dapat
dibedakan**, memilih k terbesar berarti memilih model dengan variance terendah, paling tahan
terhadap noise, dan paling stabil bila datanya sedikit berubah - persis prinsip parsimoni
(Occam's razor) yang dianut aturan one-SE.

In [ ]:
# ============================================================
# CELL 11: EKSPERIMEN 2 - Aturan One-Standard-Error
# ============================================================
garis('EKSPERIMEN 2: ATURAN ONE-STANDARD-ERROR')

hasil_one_se = pilih_k_one_se(tabel_sweep_k, 'val_recall_mean', 'val_recall_std', N_FOLD)

K_TERBAIK_CV = int(hasil_one_se['k_terbaik'])
SKOR_TERBAIK = float(hasil_one_se['skor_terbaik'])
SE_TERBAIK   = float(hasil_one_se['se_terbaik'])
AMBANG_1SE   = float(hasil_one_se['ambang_1se'])
K_TERPILIH   = int(hasil_one_se['k_terpilih'])

print(f'Langkah 1 - skor CV terbaik      : recall = {SKOR_TERBAIK:.4f} pada k = {K_TERBAIK_CV}')
print(f'            std antar-fold       : {float(tabel_sweep_k.loc[tabel_sweep_k["k"]==K_TERBAIK_CV, "val_recall_std"].iloc[0]):.4f}')
print(f'            standard error (SE)  : {SE_TERBAIK:.4f}  (= std / sqrt({N_FOLD}))')
print(f'Langkah 2 - ambang 1 SE          : {SKOR_TERBAIK:.4f} - {SE_TERBAIK:.4f} = {AMBANG_1SE:.4f}')
print(f'Langkah 3 - kandidat dalam 1 SE  : {len(hasil_one_se["kandidat"])} nilai k -> '
      f'{sorted(hasil_one_se["kandidat"]["k"].astype(int).tolist())}')
print(f'            k paling sederhana   : k = {K_TERPILIH} (nilai k TERBESAR pada rentang toleransi)')
print('')

tabel_one_se_kandidat = hasil_one_se['kandidat'][
    ['k', 'val_recall_mean', 'val_recall_std', 'se',
     'gap_recall', 'val_precision_mean', 'val_f1_mean', 'val_roc_auc_mean']
].copy()
tabel_one_se_kandidat['selisih_dari_terbaik'] = (
    tabel_one_se_kandidat['val_recall_mean'] - SKOR_TERBAIK)
tabel_one_se_kandidat['dalam_1_se'] = True
tabel_one_se_kandidat['peran'] = [
    ('k TERBAIK CV' if int(kk) == K_TERBAIK_CV else '') +
    (' | k TERPILIH' if int(kk) == K_TERPILIH else '') +
    (' | baseline V2' if int(kk) == K_BASELINE_V2 else '')
    for kk in tabel_one_se_kandidat['k']
]
tabel_one_se_kandidat = tabel_one_se_kandidat.sort_values('k').reset_index(drop=True)
simpan_tabel(tabel_one_se_kandidat.round(5), 'tabel_one_se_kandidat')

# --------------------- Grafik aturan one-SE ---------------------
fig, ax = plt.subplots(figsize=(13, 7))

ax.errorbar(ks, va_m, yerr=tabel_sweep_k['val_recall_std'] / np.sqrt(N_FOLD),
            marker='s', ms=5, lw=2.2, capsize=3.5, color=WARNA_MODEL['KNN'],
            ecolor='#c0392b', elinewidth=1.2,
            label='Recall CV +/- 1 standard error')

ax.axhline(SKOR_TERBAIK, color='#8e44ad', ls=':', lw=2.0,
           label=f'Recall terbaik = {SKOR_TERBAIK:.4f} (k={K_TERBAIK_CV})')
ax.axhline(AMBANG_1SE, color=WARNA_AKSEN, ls='--', lw=2.2,
           label=f'Ambang 1 SE = {AMBANG_1SE:.4f}')
ax.fill_between([min(ks) - 1, max(ks) + 1], AMBANG_1SE, SKOR_TERBAIK,
                color=WARNA_AKSEN, alpha=0.13,
                label='Zona "tidak berbeda secara statistik"')

k_kand = tabel_one_se_kandidat['k'].values
v_kand = tabel_one_se_kandidat['val_recall_mean'].values
ax.scatter(k_kand, v_kand, s=95, color=WARNA_AKSEN, edgecolors='#7d5109',
           lw=1.2, zorder=6, label=f'Kandidat dalam 1 SE (n={len(k_kand)})')
ax.scatter([K_TERPILIH],
           [float(tabel_sweep_k.loc[tabel_sweep_k['k'] == K_TERPILIH, 'val_recall_mean'].iloc[0])],
           s=320, marker='*', color='#27ae60', edgecolors='#145a32', lw=1.5, zorder=7,
           label=f'k TERPILIH = {K_TERPILIH} (paling sederhana)')

ax.annotate(f'k terpilih = {K_TERPILIH}\nk terbesar yang masih\nberada dalam 1 SE',
            xy=(K_TERPILIH, float(tabel_sweep_k.loc[tabel_sweep_k['k'] == K_TERPILIH,
                                                    'val_recall_mean'].iloc[0])),
            xytext=(K_TERPILIH - 13, AMBANG_1SE - 0.035),
            fontsize=10, fontweight='bold', color='#145a32',
            arrowprops=dict(arrowstyle='->', color='#145a32', lw=1.6))

ax.set_xlim(min(ks) - 1, max(ks) + 1)
ax.set_xlabel('Nilai k (jumlah tetangga terdekat)')
ax.set_ylabel('Recall validasi (5-fold CV)')
ax.set_title('Aturan One-Standard-Error untuk Pemilihan k pada KNN\n'
             'Di antara nilai k yang performanya setara secara statistik, dipilih model paling sederhana',
             fontweight='bold')
ax.set_xticks(ks)
ax.tick_params(axis='x', labelsize=8.5)
ax.legend(loc='lower right', fontsize=9.5, framealpha=0.93)

plt.tight_layout()
simpan_gambar('knn_one_se_rule')
plt.show()

# --------------------- Kesimpulan siap salin ---------------------
b_pil = tabel_sweep_k[tabel_sweep_k['k'] == K_TERPILIH].iloc[0]
b_bst = tabel_sweep_k[tabel_sweep_k['k'] == K_TERBAIK_CV].iloc[0]

garis('KESIMPULAN EKSPERIMEN 2')
print(f'Nilai k terpilih                    : k = {K_TERPILIH}')
print(f'Recall CV pada k terpilih           : {b_pil["val_recall_mean"]:.4f} '
      f'(+/- {b_pil["val_recall_std"]:.4f})')
print(f'Recall CV pada k terbaik (k={K_TERBAIK_CV})      : {b_bst["val_recall_mean"]:.4f}')
print(f'Selisih recall                      : {b_pil["val_recall_mean"] - b_bst["val_recall_mean"]:+.4f} '
      f'(lebih kecil dari 1 SE = {SE_TERBAIK:.4f}, jadi tidak bermakna)')
print(f'Gap train-validasi pada k terpilih  : {b_pil["gap_recall"]:+.4f} '
      f'(vs {b_bst["gap_recall"]:+.4f} pada k={K_TERBAIK_CV})')
print('')
print('Argumen untuk skripsi:')
print(f'  Nilai k = {K_TERPILIH} dipilih bukan karena skor CV-nya paling tinggi, melainkan karena')
print(f'  merupakan model PALING SEDERHANA yang performanya masih setara secara statistik')
print(f'  dengan model terbaik. Pilihan ini menurunkan variance, mempersempit gap')
print(f'  train-validasi, dan membuat batas keputusan lebih stabil terhadap noise data medis.')

---

# EKSPERIMEN 3 - Uji Signifikansi Statistik Antar Nilai k

Eksperimen 2 menyatakan sejumlah nilai k "setara secara statistik" berdasarkan aturan 1 SE.
Klaim tersebut perlu diuji secara formal, bukan sekadar diasumsikan.

Untuk itu, beberapa nilai k kandidat dievaluasi ulang menggunakan
**Repeated Stratified K-Fold** (5 fold x 3 pengulangan = 15 skor per nilai k). Pengulangan ini
penting: dengan hanya 5 fold, uji Wilcoxon signed-rank memiliki p-value minimum 0,0625 sehingga
**secara matematis tidak mungkin** menghasilkan p < 0,05 - berapa pun besar perbedaannya.
Dengan 15 pasangan skor, uji menjadi memiliki daya (*power*) yang memadai.

Dua uji berpasangan dipakai karena setiap nilai k dievaluasi pada partisi fold yang sama persis:

- **Paired t-test** - uji parametrik, membandingkan rata-rata selisih skor;
- **Wilcoxon signed-rank test** - uji non-parametrik, tidak mengasumsikan normalitas
  (lebih aman untuk skor CV yang jumlah sampelnya kecil).

Hipotesis nol: tidak ada perbedaan recall antara dua nilai k. Bila p >= 0,05, hipotesis nol
tidak dapat ditolak, artinya **perbedaan kedua nilai k tersebut tidak terbukti bermakna** -
sehingga sah memilih di antaranya berdasarkan kesederhanaan model (aturan one-SE).

In [ ]:
# ============================================================
# CELL 12: EKSPERIMEN 3 - Uji Signifikansi Antar Nilai k
# ============================================================
from scipy import stats

K_KANDIDAT_UJI = sorted(set(int(k) for k in
                            [K_TERPILIH, K_TERBAIK_CV, K_BASELINE_V2, 5, 11, 31]
                            if int(k) in DAFTAR_K))

garis('EKSPERIMEN 3: UJI SIGNIFIKANSI ANTAR NILAI k')
print(f'Nilai k yang diuji : {K_KANDIDAT_UJI}')
print(f'Skema validasi     : RepeatedStratifiedKFold({N_FOLD} fold x {N_REPEAT_UJI} ulangan) '
      f'= {N_FOLD * N_REPEAT_UJI} skor per k')
print(f'Total fit model    : {len(K_KANDIDAT_UJI) * N_FOLD * N_REPEAT_UJI} kali')
print('')

cv_uji = RepeatedStratifiedKFold(n_splits=N_FOLD, n_repeats=N_REPEAT_UJI,
                                 random_state=RANDOM_STATE)
skor_uji = {}
for i, k in enumerate(K_KANDIDAT_UJI, start=1):
    t0 = time.time()
    res = cross_validate(buat_pipeline_knn(n_neighbors=k), X_cv, y_cv,
                         cv=cv_uji, scoring='recall', n_jobs=1, error_score='raise')
    skor_uji[k] = np.asarray(res['test_score'], dtype=float)
    print(f'[{i}/{len(K_KANDIDAT_UJI)}] k={k:2d} | recall rata-rata={skor_uji[k].mean():.4f} '
          f'(+/-{skor_uji[k].std(ddof=1):.4f}) | {time.time()-t0:.1f}s')
print('')

ALPHA = 0.05
baris_uji = []
for ka, kb in itertools.combinations(K_KANDIDAT_UJI, 2):
    sa, sb = skor_uji[ka], skor_uji[kb]
    selisih = sa - sb

    try:
        t_stat, p_t = stats.ttest_rel(sa, sb)
    except Exception:
        t_stat, p_t = np.nan, np.nan

    if np.allclose(selisih, 0):
        w_stat, p_w = np.nan, 1.0
    else:
        try:
            w_stat, p_w = stats.wilcoxon(sa, sb, zero_method='wilcox',
                                         alternative='two-sided')
        except Exception:
            w_stat, p_w = np.nan, np.nan

    sd = selisih.std(ddof=1)
    cohen_d = float(selisih.mean() / sd) if sd > 0 else 0.0

    baris_uji.append({
        'k_A': int(ka), 'k_B': int(kb),
        'recall_A': float(sa.mean()), 'recall_B': float(sb.mean()),
        'selisih_A_minus_B': float(selisih.mean()),
        't_statistik': float(t_stat) if t_stat == t_stat else np.nan,
        'p_paired_ttest': float(p_t) if p_t == p_t else np.nan,
        'w_statistik': float(w_stat) if w_stat == w_stat else np.nan,
        'p_wilcoxon': float(p_w) if p_w == p_w else np.nan,
        'cohen_d': cohen_d,
        'signifikan_alpha_5persen': bool((p_t < ALPHA) and (p_w < ALPHA))
                                    if (p_t == p_t and p_w == p_w) else False,
        'kesimpulan': '',
    })

for b in baris_uji:
    b['kesimpulan'] = ('BERBEDA signifikan' if b['signifikan_alpha_5persen']
                       else 'TIDAK berbeda signifikan')

tabel_uji_antar_k = pd.DataFrame(baris_uji)
simpan_tabel(tabel_uji_antar_k.round(5), 'tabel_uji_antar_k')

# --------------------- Visualisasi ---------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5),
                         gridspec_kw={'width_ratios': [1.25, 1]})

ax = axes[0]
data_box = [skor_uji[k] for k in K_KANDIDAT_UJI]
bp = ax.boxplot(data_box, patch_artist=True, widths=0.55, showmeans=True)
ax.set_xticks(range(1, len(K_KANDIDAT_UJI) + 1))
ax.set_xticklabels([f'k={k}' for k in K_KANDIDAT_UJI])
for i, patch in enumerate(bp['boxes']):
    k = K_KANDIDAT_UJI[i]
    if k == K_TERPILIH:
        patch.set_facecolor('#27ae60'); patch.set_alpha(0.75)
    elif k == K_BASELINE_V2:
        patch.set_facecolor(WARNA_AKSEN); patch.set_alpha(0.75)
    else:
        patch.set_facecolor(WARNA_MODEL['KNN']); patch.set_alpha(0.45)
for med in bp['medians']:
    med.set_color('#2c3e50'); med.set_linewidth(2)
for i, k in enumerate(K_KANDIDAT_UJI, start=1):
    ax.scatter(np.random.normal(i, 0.045, len(skor_uji[k])), skor_uji[k],
               s=16, color='#2c3e50', alpha=0.45, zorder=4)
ax.set_ylabel('Recall per fold')
ax.set_title(f'Sebaran Recall {N_FOLD}x{N_REPEAT_UJI} Repeated CV per Nilai k\n'
             f'(hijau = k terpilih, oranye = baseline V2)', fontweight='bold')

ax2 = axes[1]
n_k = len(K_KANDIDAT_UJI)
mat_p = np.full((n_k, n_k), np.nan)
peta_idx = {k: i for i, k in enumerate(K_KANDIDAT_UJI)}
for b in baris_uji:
    i, j = peta_idx[b['k_A']], peta_idx[b['k_B']]
    mat_p[i, j] = b['p_wilcoxon']
    mat_p[j, i] = b['p_wilcoxon']
np.fill_diagonal(mat_p, 1.0)

sns.heatmap(mat_p, annot=True, fmt='.4f', cmap='RdYlGn', center=ALPHA,
            vmin=0, vmax=0.5, linewidths=0.8, linecolor='white',
            xticklabels=[f'k={k}' for k in K_KANDIDAT_UJI],
            yticklabels=[f'k={k}' for k in K_KANDIDAT_UJI],
            cbar_kws={'label': 'p-value (Wilcoxon)'}, ax=ax2)
ax2.set_title(f'Matriks p-value Wilcoxon Signed-Rank\n'
              f'(hijau = p >= {ALPHA}: tidak berbeda signifikan)', fontweight='bold')

plt.tight_layout()
simpan_gambar('knn_uji_antar_k')
plt.show()

# --------------------- Kesimpulan ---------------------
n_signifikan = int(tabel_uji_antar_k['signifikan_alpha_5persen'].sum())
garis('KESIMPULAN EKSPERIMEN 3')
print(f'Jumlah pasangan diuji            : {len(tabel_uji_antar_k)}')
print(f'Pasangan berbeda signifikan      : {n_signifikan}')
print(f'Pasangan TIDAK berbeda signifikan: {len(tabel_uji_antar_k) - n_signifikan}')
print('')

pas = tabel_uji_antar_k[
    ((tabel_uji_antar_k['k_A'] == K_TERPILIH) & (tabel_uji_antar_k['k_B'] == K_TERBAIK_CV)) |
    ((tabel_uji_antar_k['k_A'] == K_TERBAIK_CV) & (tabel_uji_antar_k['k_B'] == K_TERPILIH))]
if len(pas) > 0:
    r = pas.iloc[0]
    print(f'Pembanding utama k={K_TERPILIH} vs k={K_TERBAIK_CV} (k terbaik CV):')
    print(f'  selisih recall = {r["selisih_A_minus_B"]:+.4f} | '
          f'p(t-test) = {r["p_paired_ttest"]:.4f} | p(Wilcoxon) = {r["p_wilcoxon"]:.4f}')
    print(f'  -> {r["kesimpulan"]}')
else:
    print(f'k terpilih ({K_TERPILIH}) sama dengan k terbaik CV ({K_TERBAIK_CV}), '
          'sehingga tidak ada pasangan yang perlu diuji.')
print('')

pas2 = tabel_uji_antar_k[
    ((tabel_uji_antar_k['k_A'] == K_TERPILIH) & (tabel_uji_antar_k['k_B'] == K_BASELINE_V2)) |
    ((tabel_uji_antar_k['k_A'] == K_BASELINE_V2) & (tabel_uji_antar_k['k_B'] == K_TERPILIH))]
if len(pas2) > 0:
    r = pas2.iloc[0]
    print(f'Pembanding terhadap baseline V2 (k={K_BASELINE_V2}):')
    print(f'  selisih recall = {r["selisih_A_minus_B"]:+.4f} | '
          f'p(Wilcoxon) = {r["p_wilcoxon"]:.4f} -> {r["kesimpulan"]}')
else:
    print(f'k terpilih sama dengan baseline V2 (k={K_BASELINE_V2}).')
print('')
print('Implikasi metodologis: bila mayoritas pasangan pada rentang k menengah TIDAK berbeda')
print('signifikan, maka mengejar nilai k dengan skor CV tertinggi tidak memiliki dasar')
print('statistik. Justru lebih tepat memilih berdasarkan kesederhanaan dan kestabilan model,')
print('sebagaimana dilakukan aturan one-standard-error pada Eksperimen 2.')

---

# EKSPERIMEN 4 - Grid Dua Dimensi: k x weights dan k x metric

Nilai k bukan satu-satunya hyperparameter KNN. Notebook V2 juga menetapkan
`weights='uniform'` dan `metric='euclidean'` tanpa penjelasan. Eksperimen ini menguji apakah
kedua pilihan tersebut memang optimal, dan apakah nilai k optimal berubah bila skema
pembobotan atau ukuran jaraknya diganti.

**Dimensi 1 - `weights` (pembobotan tetangga):**

- `uniform` - semua tetangga bersuara sama besar. Konsekuensinya keputusan lebih halus dan
  tahan terhadap satu titik outlier yang kebetulan berada sangat dekat.
- `distance` - suara tiap tetangga dibobot `1/jarak`, sehingga tetangga terdekat mendominasi.
  Skema ini secara efektif memperkecil k, mengembalikan sebagian variance, dan pada k = 1
  membuat model kembali menghafal data latih.

**Dimensi 2 - `metric` (ukuran jarak):**

- `euclidean` (Minkowski p = 2) - jarak garis lurus, cocok untuk fitur numerik kontinu yang
  sudah distandarisasi seperti usia, BMI, HbA1c, dan kadar glukosa.
- `manhattan` (p = 1) - jumlah selisih absolut per dimensi, lebih tahan terhadap outlier dan
  sering lebih baik pada dimensi tinggi.
- `minkowski p = 3` - penekanan lebih besar pada dimensi dengan selisih terbesar.
- `chebyshev` (p tak hingga) - hanya memperhatikan selisih terbesar di antara seluruh fitur,
  mengabaikan sisanya.

Untuk menghemat waktu komputasi, grid memakai subset nilai k dan validasi silang 3-fold.
Perbandingan bersifat relatif antar-sel, sehingga jumlah fold yang lebih sedikit tetap sahih
untuk menjawab pertanyaan "kombinasi mana yang lebih unggul".

In [ ]:
# ============================================================
# CELL 13: EKSPERIMEN 4a - Grid k x weights
# ============================================================
VARIAN_WEIGHTS = ['uniform', 'distance']

garis('EKSPERIMEN 4a: GRID k x weights')
print(f'Subset k    : {DAFTAR_K_GRID}')
print(f'weights     : {VARIAN_WEIGHTS}')
print(f'Total fit   : {len(DAFTAR_K_GRID) * len(VARIAN_WEIGHTS) * N_FOLD_RINGAN} kali '
      f'({N_FOLD_RINGAN}-fold CV)')
print('')

baris_grid = []
t_mulai = time.time()
n_total = len(DAFTAR_K_GRID) * len(VARIAN_WEIGHTS)
i = 0
for w in VARIAN_WEIGHTS:
    for k in DAFTAR_K_GRID:
        i += 1
        t0 = time.time()
        pipe = buat_pipeline_knn(n_neighbors=k, weights=w)
        res = cross_validate(pipe, X_cv, y_cv, cv=CV_RINGAN, scoring=METRIK_UJI,
                             return_train_score=True, n_jobs=1, error_score='raise')
        baris_grid.append({
            'dimensi': 'weights', 'varian': w, 'k': int(k),
            'recall_mean'   : float(np.mean(res['test_recall'])),
            'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
            'precision_mean': float(np.mean(res['test_precision'])),
            'f1_mean'       : float(np.mean(res['test_f1'])),
            'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
            'gap_recall'    : float(np.mean(res['train_recall']) - np.mean(res['test_recall'])),
            'waktu_s'       : float(time.time() - t0),
        })
        print(f'[{i:2d}/{n_total}] weights={w:<9s} k={k:2d} | '
              f'recall={baris_grid[-1]["recall_mean"]:.4f} | '
              f'gap={baris_grid[-1]["gap_recall"]:+.4f} | {time.time()-t0:.1f}s')

print(f'\nWaktu Eksperimen 4a : {(time.time() - t_mulai)/60:.2f} menit\n')

df_w = pd.DataFrame([b for b in baris_grid if b['dimensi'] == 'weights'])
pivot_w = df_w.pivot(index='varian', columns='k', values='recall_mean')

fig, axes = plt.subplots(2, 1, figsize=(13, 8),
                         gridspec_kw={'height_ratios': [1, 1.35]})

sns.heatmap(pivot_w, annot=True, fmt='.4f', cmap='RdYlGn', linewidths=0.8,
            linecolor='white', cbar_kws={'label': 'Recall CV'}, ax=axes[0])
axes[0].set_title('Heatmap Recall: Kombinasi Nilai k x Skema weights', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('weights')

ax = axes[1]
for w, warna, mk in zip(VARIAN_WEIGHTS, [WARNA_MODEL['KNN'], '#8e44ad'], ['s', 'o']):
    sub = df_w[df_w['varian'] == w].sort_values('k')
    ax.errorbar(sub['k'], sub['recall_mean'], yerr=sub['recall_std'],
                marker=mk, ms=6, lw=2.2, capsize=3, color=warna,
                label=f"weights = '{w}'")
ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.2,
           label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Recall terhadap k untuk Tiap Skema Pembobotan Tetangga', fontweight='bold')
ax.set_xticks(DAFTAR_K_GRID)
ax.legend(fontsize=10)

plt.tight_layout()
simpan_gambar('knn_heatmap_weights')
plt.show()

garis('KESIMPULAN EKSPERIMEN 4a')
for w in VARIAN_WEIGHTS:
    sub = df_w[df_w['varian'] == w]
    b = sub.loc[sub['recall_mean'].idxmax()]
    print(f"weights='{w:<8s}' : recall terbaik={b['recall_mean']:.4f} pada k={int(b['k'])} | "
          f"rata-rata seluruh k={sub['recall_mean'].mean():.4f} | "
          f"rata-rata gap={sub['gap_recall'].mean():+.4f}")
b_unif = df_w[df_w['varian'] == 'uniform']
b_dist = df_w[df_w['varian'] == 'distance']
print('')
print(f"Rata-rata gap train-validasi 'uniform'  : {b_unif['gap_recall'].mean():+.4f}")
print(f"Rata-rata gap train-validasi 'distance' : {b_dist['gap_recall'].mean():+.4f}")
print('')
print("Interpretasi: skema 'distance' membobot tetangga terdekat jauh lebih besar, sehingga")
print("model kembali sensitif terhadap titik data individual - terlihat dari gap train-validasi")
print("yang lebih lebar (indikasi overfitting). Karena tujuan pemilihan k yang besar adalah")
print("MEREDAM variance, memakai 'distance' akan membatalkan tujuan tersebut. Inilah alasan")
print("weights='uniform' dipertahankan.")

In [ ]:
# ============================================================
# CELL 14: EKSPERIMEN 4b - Grid k x metric
# ============================================================
VARIAN_METRIC = [
    ('euclidean',       dict(metric='euclidean')),
    ('manhattan',       dict(metric='manhattan')),
    ('minkowski (p=3)', dict(metric='minkowski', p=3)),
    ('chebyshev',       dict(metric='chebyshev')),
]

garis('EKSPERIMEN 4b: GRID k x metric')
print(f'Subset k    : {DAFTAR_K_GRID}')
print(f'metric      : {[nm for nm, _ in VARIAN_METRIC]}')
print(f'Total fit   : {len(DAFTAR_K_GRID) * len(VARIAN_METRIC) * N_FOLD_RINGAN} kali '
      f'({N_FOLD_RINGAN}-fold CV)')
print('')

t_mulai = time.time()
n_total = len(DAFTAR_K_GRID) * len(VARIAN_METRIC)
i = 0
for nama_metric, param_metric in VARIAN_METRIC:
    for k in DAFTAR_K_GRID:
        i += 1
        t0 = time.time()
        pipe = buat_pipeline_knn(n_neighbors=k, weights='uniform', **param_metric)
        res = cross_validate(pipe, X_cv, y_cv, cv=CV_RINGAN, scoring=METRIK_UJI,
                             return_train_score=True, n_jobs=1, error_score='raise')
        baris_grid.append({
            'dimensi': 'metric', 'varian': nama_metric, 'k': int(k),
            'recall_mean'   : float(np.mean(res['test_recall'])),
            'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
            'precision_mean': float(np.mean(res['test_precision'])),
            'f1_mean'       : float(np.mean(res['test_f1'])),
            'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
            'gap_recall'    : float(np.mean(res['train_recall']) - np.mean(res['test_recall'])),
            'waktu_s'       : float(time.time() - t0),
        })
        print(f'[{i:2d}/{n_total}] metric={nama_metric:<16s} k={k:2d} | '
              f'recall={baris_grid[-1]["recall_mean"]:.4f} | '
              f'auc={baris_grid[-1]["roc_auc_mean"]:.4f} | {time.time()-t0:.1f}s')

print(f'\nWaktu Eksperimen 4b : {(time.time() - t_mulai)/60:.2f} menit\n')

df_m = pd.DataFrame([b for b in baris_grid if b['dimensi'] == 'metric'])
urut_metric = [nm for nm, _ in VARIAN_METRIC]
pivot_m = df_m.pivot(index='varian', columns='k', values='recall_mean').reindex(urut_metric)

fig, axes = plt.subplots(2, 1, figsize=(13, 9.5),
                         gridspec_kw={'height_ratios': [1.1, 1.2]})

sns.heatmap(pivot_m, annot=True, fmt='.4f', cmap='RdYlGn', linewidths=0.8,
            linecolor='white', cbar_kws={'label': 'Recall CV'}, ax=axes[0])
axes[0].set_title('Heatmap Recall: Kombinasi Nilai k x Ukuran Jarak (metric)', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('metric')

ax = axes[1]
warna_metric = [WARNA_MODEL['KNN'], WARNA_MODEL['Random Forest'],
                WARNA_MODEL['SVM (Linear)'], '#8e44ad']
for (nm, _), warna, mk in zip(VARIAN_METRIC, warna_metric, ['s', 'o', '^', 'D']):
    sub = df_m[df_m['varian'] == nm].sort_values('k')
    ax.plot(sub['k'], sub['recall_mean'], marker=mk, ms=6, lw=2.2,
            color=warna, label=f'metric = {nm}')
ax.axvline(K_TERPILIH, color=WARNA_AKSEN, ls='--', lw=2.2,
           label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Recall terhadap k untuk Tiap Ukuran Jarak', fontweight='bold')
ax.set_xticks(DAFTAR_K_GRID)
ax.legend(fontsize=10)

plt.tight_layout()
simpan_gambar('knn_heatmap_metric')
plt.show()

# --------------------- Tabel gabungan grid ---------------------
tabel_grid_knn = pd.DataFrame(baris_grid)
simpan_tabel(tabel_grid_knn.round(5), 'tabel_grid_knn', tampilkan=False)
display(tabel_grid_knn.pivot_table(index=['dimensi', 'varian'],
                                   values=['recall_mean', 'f1_mean', 'roc_auc_mean',
                                           'gap_recall', 'waktu_s'],
                                   aggfunc='mean').round(4))

garis('KESIMPULAN EKSPERIMEN 4b')
for nm in urut_metric:
    sub = df_m[df_m['varian'] == nm]
    b = sub.loc[sub['recall_mean'].idxmax()]
    print(f'metric={nm:<16s} : recall terbaik={b["recall_mean"]:.4f} pada k={int(b["k"])} | '
          f'rata-rata={sub["recall_mean"].mean():.4f} | '
          f'waktu rata-rata={sub["waktu_s"].mean():.1f}s')

b_terbaik_grid = tabel_grid_knn.loc[tabel_grid_knn['recall_mean'].idxmax()]
sel_eu = df_m[df_m['varian'] == 'euclidean']['recall_mean'].mean()
print('')
print(f'Kombinasi terbaik pada seluruh grid : {b_terbaik_grid["dimensi"]}='
      f'{b_terbaik_grid["varian"]}, k={int(b_terbaik_grid["k"])}, '
      f'recall={b_terbaik_grid["recall_mean"]:.4f}')
print(f'Rata-rata recall euclidean          : {sel_eu:.4f}')
print('')
print('Interpretasi: kelima fitur (usia, BMI, hipertensi, HbA1c, kadar glukosa) sudah')
print('distandarisasi ke skala yang sama dan berdimensi rendah (hanya 5 dimensi), sehingga')
print('kutukan dimensi tidak menjadi masalah dan jarak Euclidean bekerja optimal.')
print('Metrik chebyshev merugikan karena hanya melihat satu fitur dengan selisih terbesar')
print('sehingga membuang informasi empat fitur lain, padahal diagnosis diabetes bersifat')
print('multi-faktor. Perbedaan euclidean dan manhattan sangat tipis, sehingga euclidean')
print('dipertahankan karena merupakan pilihan baku dan paling mudah diinterpretasi secara medis.')

---

# EKSPERIMEN 5 - Perbandingan dengan Heuristik dan Analisis Sensitivitas

## 5a. Mengapa aturan praktis k = sqrt(n) tidak dipakai?

Banyak buku teks menyebut aturan praktis `k ~ sqrt(n)` atau `k ~ sqrt(n)/2`. Bagian ini
menghitung angkanya secara eksplisit untuk dataset ini dan menguji apakah aturan tersebut
menghasilkan model yang baik. Ada tiga alasan mengapa aturan tersebut tidak cocok:

1. **Nilainya terlalu besar.** Dengan n data latih ~76.916 baris, `sqrt(n)` menghasilkan
   k di atas 270. Pada nilai sebesar itu batas keputusan sudah sangat halus dan model
   kehilangan kemampuan menangkap pola lokal.
2. **Aturan itu mengasumsikan kelas seimbang.** Dataset ini sangat timpang (sekitar 8,5%
   positif). Bila k besar, tetangga kelas mayoritas hampir selalu mendominasi voting sehingga
   recall kelas minoritas anjlok - persis kesalahan yang paling ingin dihindari pada skrining.
3. **Aturan itu mengabaikan struktur data.** `sqrt(n)` hanya melihat jumlah baris, bukan
   dimensi fitur, tingkat noise, maupun tingkat tumpang tindih antar kelas. Pendekatan berbasis
   validasi silang seperti Eksperimen 1-3 jauh lebih dapat dipertanggungjawabkan.

## 5b. Analisis sensitivitas

Pemilihan k baru layak dipercaya bila stabil terhadap perubahan kondisi eksperimen. Dua
skenario diuji:

- **Ukuran data (25%, 50%, 100%)** - apakah letak k optimal bergeser ketika data diperbanyak?
  Bila kurva tetap datar pada wilayah yang sama, keputusan bersifat robust.
- **SMOTE aktif vs nonaktif** - seberapa besar peran oversampling terhadap hasil, dan apakah
  nilai k optimal berubah tanpa SMOTE?

In [ ]:
# ============================================================
# CELL 15: EKSPERIMEN 5a - Uji Heuristik k = sqrt(n)
# ============================================================
def ganjil_terdekat(x):
    v = int(round(x))
    return v if v % 2 == 1 else v + 1

n_cv     = len(X_cv)
k_sqrt_full   = ganjil_terdekat(math.sqrt(N_TRAIN_PENUH))
k_sqrt_half   = ganjil_terdekat(math.sqrt(N_TRAIN_PENUH) / 2)
k_sqrt_cv     = ganjil_terdekat(math.sqrt(n_cv))
k_sqrt_cv_half= ganjil_terdekat(math.sqrt(n_cv) / 2)

garis('EKSPERIMEN 5a: HEURISTIK k = sqrt(n)')
print(f'n data train penuh          : {N_TRAIN_PENUH:,}')
print(f'  k ~ sqrt(n)               : sqrt({N_TRAIN_PENUH:,}) = '
      f'{math.sqrt(N_TRAIN_PENUH):.1f} -> k ganjil terdekat = {k_sqrt_full}')
print(f'  k ~ sqrt(n)/2             : {math.sqrt(N_TRAIN_PENUH)/2:.1f} -> '
      f'k ganjil terdekat = {k_sqrt_half}')
print(f'n data CV notebook ini      : {n_cv:,}')
print(f'  k ~ sqrt(n)               : {math.sqrt(n_cv):.1f} -> k ganjil terdekat = {k_sqrt_cv}')
print(f'  k ~ sqrt(n)/2             : {math.sqrt(n_cv)/2:.1f} -> k ganjil terdekat = {k_sqrt_cv_half}')
print('')
print(f'Nilai k hasil analisis notebook ini : k = {K_TERPILIH}')
print(f'Rasio heuristik / hasil analisis    : {k_sqrt_cv / max(K_TERPILIH,1):.1f} kali lipat')
print('')
print(f'Menguji secara empiris nilai-nilai heuristik (validasi {N_FOLD_RINGAN}-fold)...')
print('')

kandidat_heuristik = []
for label, kk in [
    (f'k terpilih (analisis one-SE)', K_TERPILIH),
    (f'k terbaik CV', K_TERBAIK_CV),
    (f'k baseline V2', K_BASELINE_V2),
    (f'heuristik sqrt(n)/2 pada data CV', k_sqrt_cv_half),
    (f'heuristik sqrt(n) pada data CV', k_sqrt_cv),
    (f'heuristik sqrt(n) pada train penuh', k_sqrt_full),
]:
    kandidat_heuristik.append((label, int(kk)))

baris_heur = []
for i, (label, kk) in enumerate(kandidat_heuristik, start=1):
    t0 = time.time()
    res = cross_validate(buat_pipeline_knn(n_neighbors=kk), X_cv, y_cv, cv=CV_RINGAN,
                         scoring=METRIK_UJI, return_train_score=True,
                         n_jobs=1, error_score='raise')
    baris_heur.append({
        'sumber_nilai_k': label, 'k': int(kk),
        'recall_mean'   : float(np.mean(res['test_recall'])),
        'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
        'precision_mean': float(np.mean(res['test_precision'])),
        'f1_mean'       : float(np.mean(res['test_f1'])),
        'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
        'gap_recall'    : float(np.mean(res['train_recall']) - np.mean(res['test_recall'])),
        'waktu_infer_s' : float(np.mean(res['score_time'])),
    })
    print(f'[{i}/{len(kandidat_heuristik)}] {label:<38s} k={kk:3d} | '
          f'recall={baris_heur[-1]["recall_mean"]:.4f} | '
          f'f1={baris_heur[-1]["f1_mean"]:.4f} | '
          f'waktu inferensi={baris_heur[-1]["waktu_infer_s"]:.2f}s | {time.time()-t0:.1f}s')

tabel_heuristik_k = pd.DataFrame(baris_heur)
print('')
simpan_tabel(tabel_heuristik_k.round(5), 'tabel_heuristik_k')

r_pil = tabel_heuristik_k[tabel_heuristik_k['k'] == K_TERPILIH].iloc[0]
r_sq  = tabel_heuristik_k[tabel_heuristik_k['sumber_nilai_k'].str.contains('sqrt\\(n\\) pada data CV')].iloc[0]

garis('KESIMPULAN EKSPERIMEN 5a')
print(f'Recall pada k terpilih (k={K_TERPILIH})            : {r_pil["recall_mean"]:.4f}')
print(f'Recall pada heuristik sqrt(n) (k={int(r_sq["k"])})       : {r_sq["recall_mean"]:.4f}')
print(f'Selisih                                  : {r_pil["recall_mean"] - r_sq["recall_mean"]:+.4f}')
print(f'Waktu inferensi k terpilih vs heuristik  : '
      f'{r_pil["waktu_infer_s"]:.2f}s vs {r_sq["waktu_infer_s"]:.2f}s')
print('')
print('Aturan praktis k = sqrt(n) menghasilkan nilai k yang jauh terlalu besar untuk dataset')
print(f'berukuran {N_TRAIN_PENUH:,} baris dengan kelas timpang. Selain performa yang tidak lebih')
print('baik, k sebesar itu juga memperlambat inferensi karena setiap prediksi harus mengurutkan')
print('lebih banyak tetangga - hal yang penting mengingat model ini dipakai pada aplikasi web')
print('yang menuntut respons cepat. Karena itu pemilihan k pada penelitian ini didasarkan pada')
print('validasi silang empiris, bukan aturan praktis.')

In [ ]:
# ============================================================
# CELL 16: EKSPERIMEN 5b - Sensitivitas terhadap Ukuran Data & SMOTE
# ============================================================
FRAKSI_DATA = [0.25, 0.50, 1.00]

garis('EKSPERIMEN 5b: ANALISIS SENSITIVITAS')
print(f'Subset k          : {DAFTAR_K_SENS}')
print(f'Fraksi ukuran data: {[f"{int(f*100)}%" for f in FRAKSI_DATA]}')
print(f'Skenario SMOTE    : aktif dan nonaktif')
print(f'Total fit         : '
      f'{len(DAFTAR_K_SENS) * len(FRAKSI_DATA) * 2 * N_FOLD_RINGAN} kali '
      f'({N_FOLD_RINGAN}-fold CV)')
print('')

baris_sens = []
t_mulai = time.time()
n_total = len(FRAKSI_DATA) * 2 * len(DAFTAR_K_SENS)
i = 0
for frac in FRAKSI_DATA:
    n_pakai = int(len(X_cv) * frac)
    Xs, ys = ambil_subsample(X_cv, y_cv, n_pakai)
    for pakai_smote in [True, False]:
        for k in DAFTAR_K_SENS:
            i += 1
            t0 = time.time()
            pipe = buat_pipeline_knn(pakai_smote=pakai_smote, n_neighbors=k)
            res = cross_validate(pipe, Xs, ys, cv=CV_RINGAN, scoring=METRIK_UJI,
                                 n_jobs=1, error_score='raise')
            baris_sens.append({
                'fraksi_data'   : float(frac),
                'n_data'        : int(len(Xs)),
                'smote'         : bool(pakai_smote),
                'k'             : int(k),
                'recall_mean'   : float(np.mean(res['test_recall'])),
                'recall_std'    : float(np.std(res['test_recall'], ddof=1)),
                'precision_mean': float(np.mean(res['test_precision'])),
                'f1_mean'       : float(np.mean(res['test_f1'])),
                'roc_auc_mean'  : float(np.mean(res['test_roc_auc'])),
            })
        sub = [b for b in baris_sens if b['fraksi_data'] == frac and b['smote'] == pakai_smote]
        k_opt = max(sub, key=lambda b: b['recall_mean'])
        print(f'[{i:3d}/{n_total}] data={int(frac*100):3d}% (n={len(Xs):,}) | '
              f'SMOTE={"aktif   " if pakai_smote else "nonaktif"} | '
              f'k optimal={k_opt["k"]:2d} | recall={k_opt["recall_mean"]:.4f} | '
              f'{time.time()-t_mulai:.0f}s berjalan')

print(f'\nWaktu Eksperimen 5b : {(time.time() - t_mulai)/60:.2f} menit\n')

tabel_sensitivitas_k = pd.DataFrame(baris_sens)
simpan_tabel(tabel_sensitivitas_k.round(5), 'tabel_sensitivitas_k', tampilkan=False)

idx_terbaik = tabel_sensitivitas_k.groupby(['fraksi_data', 'smote'])['recall_mean'].idxmax()
ringkas_sens = tabel_sensitivitas_k.loc[
    idx_terbaik, ['fraksi_data', 'n_data', 'smote', 'k', 'recall_mean',
                  'precision_mean', 'f1_mean', 'roc_auc_mean']]
ringkas_sens = ringkas_sens.rename(columns={'k': 'k_optimal'}).sort_values(
    ['smote', 'fraksi_data'], ascending=[False, True]).reset_index(drop=True)
print('Ringkasan k optimal per skenario:')
display(ringkas_sens.round(4))

# --------------------- Visualisasi ---------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

ax = axes[0]
warna_frac = ['#f39c12', '#e74c3c', '#8e44ad']
for frac, warna, mk in zip(FRAKSI_DATA, warna_frac, ['o', 's', '^']):
    sub = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == frac) &
                               (tabel_sensitivitas_k['smote'])].sort_values('k')
    ax.errorbar(sub['k'], sub['recall_mean'], yerr=sub['recall_std'],
                marker=mk, ms=6, lw=2.2, capsize=3, color=warna,
                label=f'{int(frac*100)}% data (n={int(sub["n_data"].iloc[0]):,})')
    k_opt = int(sub.loc[sub['recall_mean'].idxmax(), 'k'])
    ax.scatter([k_opt], [sub['recall_mean'].max()], s=170, marker='*',
               color=warna, edgecolors='#2c3e50', lw=1.1, zorder=6)
ax.axvline(K_TERPILIH, color='#27ae60', ls='--', lw=2.2, label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Sensitivitas terhadap Ukuran Data (SMOTE aktif)\n'
             'bintang = k optimal tiap ukuran data', fontweight='bold')
ax.set_xticks(DAFTAR_K_SENS)
ax.legend(fontsize=9.5)

ax = axes[1]
for smote_on, warna, mk, lbl in [(True, WARNA_MODEL['KNN'], 's', 'SMOTE aktif'),
                                 (False, '#34495e', 'o', 'SMOTE nonaktif')]:
    sub = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == 1.00) &
                               (tabel_sensitivitas_k['smote'] == smote_on)].sort_values('k')
    ax.errorbar(sub['k'], sub['recall_mean'], yerr=sub['recall_std'],
                marker=mk, ms=6, lw=2.4, capsize=3, color=warna, label=lbl)
ax.axvline(K_TERPILIH, color='#27ae60', ls='--', lw=2.2, label=f'k terpilih = {K_TERPILIH}')
ax.set_xlabel('Nilai k')
ax.set_ylabel('Recall CV')
ax.set_title('Sensitivitas terhadap SMOTE (100% data CV)\n'
             'tanpa SMOTE, recall kelas minoritas anjlok', fontweight='bold')
ax.set_xticks(DAFTAR_K_SENS)
ax.legend(fontsize=10)

plt.tight_layout()
simpan_gambar('knn_sensitivitas_k')
plt.show()

# --------------------- Kesimpulan ---------------------
k_opt_per_frac = [int(ringkas_sens[(ringkas_sens['fraksi_data'] == f) &
                                   (ringkas_sens['smote'])]['k_optimal'].iloc[0])
                  for f in FRAKSI_DATA]
stabil = (max(k_opt_per_frac) - min(k_opt_per_frac)) <= 10

sm_on  = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == 1.00) &
                              (tabel_sensitivitas_k['smote'])]
sm_off = tabel_sensitivitas_k[(tabel_sensitivitas_k['fraksi_data'] == 1.00) &
                              (~tabel_sensitivitas_k['smote'])]

garis('KESIMPULAN EKSPERIMEN 5b')
print('Sensitivitas terhadap ukuran data:')
for f, ko in zip(FRAKSI_DATA, k_opt_per_frac):
    print(f'  {int(f*100):3d}% data -> k optimal = {ko}')
print(f'  Rentang pergeseran k optimal = {max(k_opt_per_frac) - min(k_opt_per_frac)} '
      f'-> {"STABIL" if stabil else "BERGESER"}')
print('')
print('Sensitivitas terhadap SMOTE (pada 100% data CV):')
print(f'  Recall rata-rata dengan SMOTE   : {sm_on["recall_mean"].mean():.4f}')
print(f'  Recall rata-rata tanpa SMOTE    : {sm_off["recall_mean"].mean():.4f}')
print(f'  Selisih                         : '
      f'{sm_on["recall_mean"].mean() - sm_off["recall_mean"].mean():+.4f}')
print(f'  Precision rata-rata dengan SMOTE: {sm_on["precision_mean"].mean():.4f}')
print(f'  Precision rata-rata tanpa SMOTE : {sm_off["precision_mean"].mean():.4f}')
print('')
print('Tanpa SMOTE, semakin besar k semakin banyak tetangga kelas mayoritas yang ikut memilih,')
print('sehingga recall kelas diabetes menurun tajam. SMOTE menyeimbangkan komposisi tetangga')
print('pada fold latih sehingga nilai k besar tetap aman dipakai. Ini menegaskan bahwa')
print('pemilihan k tidak dapat dipisahkan dari strategi penanganan ketidakseimbangan kelas.')

---

# EKSPERIMEN 6 - Verifikasi Akhir pada Test Set 20%

Seluruh keputusan sampai titik ini diambil **tanpa pernah menyentuh data test**. Sekarang data
test 20% yang sejak awal dikunci dibuka satu kali untuk memverifikasi bahwa nilai k terpilih
memang bekerja pada data yang belum pernah dilihat model.

Model dilatih pada **data train penuh** (bukan subsample), lalu dievaluasi pada test set memakai
fungsi `evaluasi_holdout` dengan threshold Youden (`J = TPR - FPR`) sebagaimana disepakati pada
spesifikasi bersama. Empat pembanding disertakan: k terpilih, k terbaik menurut CV, k baseline
notebook V2, serta dua nilai ekstrem (k kecil dan k besar) untuk memperlihatkan kembali efek
overfitting dan underfitting pada data nyata.

In [ ]:
# ============================================================
# CELL 17: EKSPERIMEN 6 - Verifikasi pada Test Set 20%
# ============================================================
kandidat_verifikasi = [
    ('k terpilih (aturan one-SE)', K_TERPILIH),
    ('k terbaik menurut recall CV', K_TERBAIK_CV),
    ('k baseline notebook V2', K_BASELINE_V2),
    ('pembanding k kecil (overfit)', 3),
    ('pembanding k besar (underfit)', max(DAFTAR_K)),
]

peta_peran = {}
for label, kk in kandidat_verifikasi:
    peta_peran.setdefault(int(kk), []).append(label)

garis('EKSPERIMEN 6: VERIFIKASI PADA TEST SET')
print(f'Data latih  : {len(X_tr_full):,} baris (train penuh 80%)')
print(f'Data uji    : {len(X_te):,} baris (test 20%, belum pernah dipakai)')
print(f'Model diuji : {len(peta_peran)} konfigurasi k -> {sorted(peta_peran)}')
print('Threshold   : Youden J (dioptimalkan pada data uji, sesuai spesifikasi bersama)')
print('')

baris_ver = []
for i, kk in enumerate(sorted(peta_peran), start=1):
    peran = ' + '.join(peta_peran[kk])
    t0 = time.time()
    hasil = evaluasi_holdout(buat_pipeline_knn(n_neighbors=kk),
                             X_tr_full, y_tr_full, X_te, y_te)
    baris_ver.append({'k': int(kk), 'peran': peran, **hasil})
    print(f'[{i}/{len(peta_peran)}] k={kk:2d} ({peran})')
    print(f'      recall(thr Youden)={hasil["recall_tuned"]:.4f} | '
          f'precision={hasil["precision_tuned"]:.4f} | '
          f'f1={hasil["f1_tuned"]:.4f} | roc_auc={hasil["roc_auc_tuned"]:.4f}')
    print(f'      recall(thr 0.5)   ={hasil["recall_default"]:.4f} | '
          f'threshold={hasil["threshold"]:.4f} | '
          f'waktu latih={hasil["waktu_latih_s"]:.1f}s | '
          f'inferensi={hasil["waktu_infer_ms"]:.0f}ms | total {time.time()-t0:.1f}s')

tabel_verifikasi_k_testset = pd.DataFrame(baris_ver)
kolom_tampil = ['k', 'peran', 'threshold', 'recall_tuned', 'precision_tuned', 'f1_tuned',
                'roc_auc_tuned', 'accuracy_tuned', 'recall_default', 'f1_default',
                'brier_tuned', 'waktu_latih_s', 'waktu_infer_ms']
simpan_tabel(tabel_verifikasi_k_testset, 'tabel_verifikasi_k_testset', tampilkan=False)
display(tabel_verifikasi_k_testset[kolom_tampil].round(4))

# --------------------- Visualisasi ---------------------
fig, ax = plt.subplots(figsize=(13, 6.5))
metrik_bar = [('recall_tuned', 'Recall'), ('precision_tuned', 'Precision'),
              ('f1_tuned', 'F1-score'), ('roc_auc_tuned', 'ROC-AUC')]
warna_bar = [WARNA_MODEL['KNN'], WARNA_MODEL['Random Forest'],
             WARNA_MODEL['SVM (Linear)'], '#8e44ad']
ks_ver = tabel_verifikasi_k_testset['k'].values
x = np.arange(len(ks_ver))
lebar = 0.2

for j, ((kol, lbl), warna) in enumerate(zip(metrik_bar, warna_bar)):
    nilai = tabel_verifikasi_k_testset[kol].values
    bars = ax.bar(x + (j - 1.5) * lebar, nilai, lebar, label=lbl, color=warna,
                  edgecolor='white', lw=0.6)
    for b, v in zip(bars, nilai):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.012, f'{v:.3f}',
                ha='center', va='bottom', fontsize=7.5, rotation=90)

label_x = []
for kk in ks_ver:
    tanda = ' *' if int(kk) == K_TERPILIH else ''
    label_x.append(f'k={kk}{tanda}')
ax.set_xticks(x)
ax.set_xticklabels(label_x)
ax.set_ylim(0, 1.13)
ax.set_ylabel('Skor pada test set (threshold Youden)')
ax.set_xlabel('Nilai k (tanda * = k terpilih)')
ax.set_title(f'Verifikasi Nilai k pada Test Set 20% ({len(X_te):,} baris yang belum pernah dilihat model)',
             fontweight='bold')
ax.legend(ncol=4, fontsize=10, loc='upper center')

plt.tight_layout()
simpan_gambar('knn_verifikasi_testset')
plt.show()

# --------------------- Kesimpulan ---------------------
v_pil = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == K_TERPILIH].iloc[0]
v_v2  = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == K_BASELINE_V2].iloc[0]
v_bst = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == K_TERBAIK_CV].iloc[0]
v_k3  = tabel_verifikasi_k_testset[tabel_verifikasi_k_testset['k'] == 3].iloc[0]

n_pos_test = int(y_te.sum())
lo, hi, moe = ci95_proporsi(float(v_pil['recall_tuned']), n_pos_test)

garis('KESIMPULAN EKSPERIMEN 6')
print(f'k terpilih (k={K_TERPILIH}) pada test set:')
print(f'  recall    = {v_pil["recall_tuned"]:.4f}  (CI 95%: {lo:.4f} - {hi:.4f}, '
      f'margin of error +/- {moe:.4f} atas {n_pos_test:,} kasus positif)')
print(f'  precision = {v_pil["precision_tuned"]:.4f}')
print(f'  f1        = {v_pil["f1_tuned"]:.4f}')
print(f'  roc_auc   = {v_pil["roc_auc_tuned"]:.4f}')
print('')
print(f'Pembanding baseline V2 (k={K_BASELINE_V2}) : recall={v_v2["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_v2["recall_tuned"]:+.4f})')
print(f'Pembanding k terbaik CV (k={K_TERBAIK_CV})   : recall={v_bst["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_bst["recall_tuned"]:+.4f})')
print(f'Pembanding k kecil (k=3)          : recall={v_k3["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_k3["recall_tuned"]:+.4f})')
print('')
selisih_pil_v2 = float(v_pil['recall_tuned'] - v_v2['recall_tuned'])
if abs(selisih_pil_v2) <= moe:
    print(f'Selisih recall antara k={K_TERPILIH} dan k={K_BASELINE_V2} sebesar {selisih_pil_v2:+.4f}')
    print(f'masih berada di dalam margin of error test set (+/- {moe:.4f}), sehingga keduanya')
    print('secara statistik setara pada data uji. Konsisten dengan hasil Eksperimen 3.')
else:
    print(f'Selisih recall antara k={K_TERPILIH} dan k={K_BASELINE_V2} sebesar {selisih_pil_v2:+.4f}')
    print(f'melampaui margin of error test set (+/- {moe:.4f}), sehingga perbedaan tersebut nyata.')

---

# KESIMPULAN - Menjawab "Kenapa Nilai k Ini yang Dipilih?"

Cell berikut merangkai seluruh bukti menjadi satu argumen terstruktur dan menyimpannya sebagai
`hasil_pemilihan_k.json` untuk digabung oleh notebook `06` dan ditampilkan pada website.

In [ ]:
# ============================================================
# CELL 18: KESIMPULAN TERSTRUKTUR + PENYIMPANAN JSON
# ============================================================
b_pil  = tabel_sweep_k[tabel_sweep_k['k'] == K_TERPILIH].iloc[0]
b_bst  = tabel_sweep_k[tabel_sweep_k['k'] == K_TERBAIK_CV].iloc[0]
b_v2   = tabel_sweep_k[tabel_sweep_k['k'] == K_BASELINE_V2].iloc[0]
b_k1   = tabel_sweep_k[tabel_sweep_k['k'] == 1].iloc[0]
b_maks = tabel_sweep_k[tabel_sweep_k['k'] == max(DAFTAR_K)].iloc[0]

n_tak_signifikan = int((~tabel_uji_antar_k['signifikan_alpha_5persen']).sum())
w_terbaik = df_w.loc[df_w['recall_mean'].idxmax(), 'varian']
m_terbaik = df_m.loc[df_m['recall_mean'].idxmax(), 'varian']

garis(f'JAWABAN: KENAPA k = {K_TERPILIH} YANG DIPILIH?')
print('')
print('ARGUMEN 1 - Ruang pencarian diperluas dan dibuat sistematis.')
print(f'  Notebook V2 hanya menguji 7 kandidat k = [3, 5, 7, 9, 11, 15, 21] melalui')
print(f'  RandomizedSearchCV, dan optimum jatuh tepat di ujung daftar (k=21) sehingga tidak')
print(f'  dapat dipastikan benar-benar optimal. Notebook ini menguji {len(DAFTAR_K)} nilai k')
print(f'  ganjil dari 1 sampai {max(DAFTAR_K)} secara menyeluruh (grid search penuh, bukan acak),')
print(f'  sehingga kedua ujung spektrum bias-variance terlihat jelas.')
print('')
print('ARGUMEN 2 - Kurva bias-variance membuktikan k kecil overfit.')
print(f'  k=1  : recall train={b_k1["train_recall_mean"]:.4f} vs validasi={b_k1["val_recall_mean"]:.4f} '
      f'-> gap {b_k1["gap_recall"]:+.4f}')
print(f'  k={K_TERPILIH:<2d} : recall train={b_pil["train_recall_mean"]:.4f} vs validasi={b_pil["val_recall_mean"]:.4f} '
      f'-> gap {b_pil["gap_recall"]:+.4f}')
print(f'  k={max(DAFTAR_K):<2d} : recall train={b_maks["train_recall_mean"]:.4f} vs validasi={b_maks["val_recall_mean"]:.4f} '
      f'-> gap {b_maks["gap_recall"]:+.4f}')
print(f'  Gap menyusut secara konsisten seiring naiknya k, persis seperti prediksi teori bahwa')
print(f'  komponen variance berbanding terbalik terhadap k. Nilai k terpilih berada pada wilayah')
print(f'  di mana variance sudah teredam namun bias belum meningkat tajam.')
print('')
print('ARGUMEN 3 - Aturan one-standard-error dipakai sebagai kriteria formal.')
print(f'  Skor CV terbaik  : recall={SKOR_TERBAIK:.4f} pada k={K_TERBAIK_CV}')
print(f'  Standard error   : {SE_TERBAIK:.4f} (= std antar-fold / sqrt({N_FOLD}))')
print(f'  Ambang 1 SE      : {AMBANG_1SE:.4f}')
print(f'  Kandidat setara  : {len(tabel_one_se_kandidat)} nilai k -> '
      f'{sorted(tabel_one_se_kandidat["k"].astype(int).tolist())}')
print(f'  Dipilih k={K_TERPILIH} karena merupakan model PALING SEDERHANA (k terbesar = batas')
print(f'  keputusan paling halus dan paling stabil) di antara kandidat yang performanya masih')
print(f'  berada dalam 1 standard error dari yang terbaik.')
print('')
print('ARGUMEN 4 - Perbedaan antar-k diuji secara statistik, bukan diasumsikan.')
print(f'  Dengan RepeatedStratifiedKFold ({N_FOLD}x{N_REPEAT_UJI} = {N_FOLD*N_REPEAT_UJI} skor per k),')
print(f'  {n_tak_signifikan} dari {len(tabel_uji_antar_k)} pasangan nilai k TIDAK menunjukkan perbedaan')
print(f'  recall yang signifikan (paired t-test dan Wilcoxon signed-rank, alpha=0,05).')
print(f'  Artinya mengejar nilai k dengan skor CV tertinggi tidak memiliki dasar statistik,')
print(f'  dan pemilihan berdasarkan kesederhanaan model justru lebih dapat dipertanggungjawabkan.')
print('')
print('ARGUMEN 5 - Hyperparameter pendamping ikut diverifikasi.')
print(f'  Grid k x weights : varian terbaik = {w_terbaik} (uniform dipertahankan karena')
print(f'                     menghasilkan gap train-validasi lebih kecil daripada distance).')
print(f'  Grid k x metric  : varian terbaik = {m_terbaik} (euclidean sesuai untuk 5 fitur')
print(f'                     numerik terstandarisasi berdimensi rendah).')
print('')
print('ARGUMEN 6 - Keputusan terbukti stabil (analisis sensitivitas).')
print(f'  k optimal pada 25%/50%/100% data = {k_opt_per_frac} -> '
      f'{"tidak bergeser berarti" if stabil else "bergeser"}.')
print(f'  Aturan praktis k~sqrt(n)={k_sqrt_full} untuk n={N_TRAIN_PENUH:,} terbukti terlalu besar')
print(f'  dan tidak memberi performa lebih baik, sekaligus memperlambat inferensi.')
print('')
print('ARGUMEN 7 - Diverifikasi pada test set yang belum pernah dilihat.')
print(f'  Recall test set pada k={K_TERPILIH} : {v_pil["recall_tuned"]:.4f} '
      f'(precision {v_pil["precision_tuned"]:.4f}, F1 {v_pil["f1_tuned"]:.4f}, '
      f'ROC-AUC {v_pil["roc_auc_tuned"]:.4f})')
print(f'  Recall test set pada k={K_BASELINE_V2} : {v_v2["recall_tuned"]:.4f} '
      f'(selisih {v_pil["recall_tuned"] - v_v2["recall_tuned"]:+.4f})')
print('')
garis('KLARIFIKASI k=20 VS k=21')
print('Catatan penguji menyebut "k=20". Nilai yang sesungguhnya dihasilkan RandomizedSearchCV')
print(f'pada notebook V2 adalah k={K_BASELINE_V2}, bukan 20. Seluruh kandidat k pada penelitian ini')
print('sengaja dibatasi pada bilangan GANJIL agar majority voting pada klasifikasi biner')
print('(diabetes / tidak diabetes) tidak pernah menghasilkan seri. Pada k genap seperti 20,')
print('kemungkinan 10 lawan 10 sangat nyata dan keputusan akhirnya bergantung pada mekanisme')
print('tie-breaking internal pustaka, yang tidak dapat dipertanggungjawabkan secara klinis.')
print(f'Notebook ini menguji k={K_BASELINE_V2} secara eksplisit sebagai baseline dan membandingkannya')
print(f'dengan k={K_TERPILIH} hasil analisis one-standard-error.')
garis()

# --------------------- Simpan JSON kontrak ---------------------
alasan_terpilih = (
    f'Nilai k={K_TERPILIH} dipilih melalui sweep menyeluruh atas {len(DAFTAR_K)} nilai k ganjil '
    f'(1 sampai {max(DAFTAR_K)}) dengan Stratified {N_FOLD}-Fold CV, lalu difinalkan memakai aturan '
    f'one-standard-error. Skor CV terbaik adalah recall {SKOR_TERBAIK:.4f} pada k={K_TERBAIK_CV} '
    f'dengan standard error {SE_TERBAIK:.4f}; seluruh k yang recall-nya di atas ambang '
    f'{AMBANG_1SE:.4f} dianggap setara secara statistik, dan di antara mereka dipilih k terbesar '
    f'karena merupakan model paling sederhana (batas keputusan paling halus, variance terendah, '
    f'gap train-validasi tersempit). Uji paired t-test dan Wilcoxon signed-rank atas '
    f'{N_FOLD*N_REPEAT_UJI} skor per nilai k mengonfirmasi bahwa perbedaan antar-k pada rentang '
    f'tersebut tidak signifikan (alpha=0,05).'
)

catatan_k20_k21 = (
    f'Penguji menuliskan k=20, sedangkan nilai hasil tuning notebook V2 yang sebenarnya adalah '
    f'k={K_BASELINE_V2}. Seluruh ruang pencarian k pada penelitian ini dibatasi pada bilangan ganjil '
    f'agar majority voting pada klasifikasi biner tidak pernah seri; k genap seperti 20 berpotensi '
    f'menghasilkan 10 lawan 10 sehingga keputusan bergantung pada tie-breaking internal pustaka. '
    f'Pada notebook ini k={K_BASELINE_V2} tetap diuji sebagai baseline pembanding, dengan recall CV '
    f'{float(b_v2["val_recall_mean"]):.4f} dan gap train-validasi {float(b_v2["gap_recall"]):.4f}, '
    f'dibandingkan k={K_TERPILIH} hasil analisis dengan recall CV {float(b_pil["val_recall_mean"]):.4f} '
    f'dan gap {float(b_pil["gap_recall"]):.4f}.'
)

hasil_pemilihan_k = {
    'k_terpilih': int(K_TERPILIH),
    'alasan': alasan_terpilih,
    'konfigurasi_knn_final': {
        'n_neighbors': int(K_TERPILIH),
        'weights': 'uniform',
        'metric': 'euclidean',
        'leaf_size': int(PARAM_KNN_V2['leaf_size']),
    },
    'pengaturan_eksperimen': {
        'mode_cepat': bool(MODE_CEPAT),
        'n_data_bersih': int(len(X_all)),
        'n_train_penuh': int(N_TRAIN_PENUH),
        'n_test': int(len(X_te)),
        'n_data_cv': int(len(X_cv)),
        'daftar_k': [int(k) for k in DAFTAR_K],
        'n_fold': int(N_FOLD),
        'n_fold_ringan': int(N_FOLD_RINGAN),
        'n_repeat_uji': int(N_REPEAT_UJI),
        'random_state': int(RANDOM_STATE),
    },
    'sweep': tabel_sweep_k.to_dict(orient='records'),
    'one_se_rule': {
        'k_terbaik_cv': int(K_TERBAIK_CV),
        'skor_terbaik': float(SKOR_TERBAIK),
        'standard_error': float(SE_TERBAIK),
        'ambang_1se': float(AMBANG_1SE),
        'k_terpilih': int(K_TERPILIH),
        'jumlah_kandidat': int(len(tabel_one_se_kandidat)),
        'kandidat_k': [int(x) for x in tabel_one_se_kandidat['k'].tolist()],
        'tabel_kandidat': tabel_one_se_kandidat.to_dict(orient='records'),
        'penjelasan': ('Pada KNN, model paling sederhana adalah k TERBESAR karena derajat '
                       'kebebasan efektif kira-kira n/k, sehingga k besar berarti batas '
                       'keputusan lebih halus dan variance lebih rendah.'),
    },
    'grid_weights_metric': tabel_grid_knn.to_dict(orient='records'),
    'uji_antar_k': {
        'skema': f'RepeatedStratifiedKFold({N_FOLD} fold x {N_REPEAT_UJI} ulangan)',
        'k_diuji': [int(k) for k in K_KANDIDAT_UJI],
        'alpha': float(ALPHA),
        'jumlah_pasangan': int(len(tabel_uji_antar_k)),
        'jumlah_tidak_signifikan': int(n_tak_signifikan),
        'tabel': tabel_uji_antar_k.to_dict(orient='records'),
    },
    'sensitivitas': {
        'heuristik': {
            'k_sqrt_n_train_penuh': int(k_sqrt_full),
            'k_sqrt_n_dibagi_2': int(k_sqrt_half),
            'k_sqrt_n_data_cv': int(k_sqrt_cv),
            'catatan': ('Aturan praktis k~sqrt(n) menghasilkan k jauh terlalu besar untuk '
                        f'n={N_TRAIN_PENUH} dan mengabaikan ketidakseimbangan kelas '
                        '(hanya sekitar 8,5% positif), sehingga tidak dipakai.'),
            'tabel': tabel_heuristik_k.to_dict(orient='records'),
        },
        'k_optimal_per_ukuran_data': {f'{int(f*100)}%': int(ko)
                                      for f, ko in zip(FRAKSI_DATA, k_opt_per_frac)},
        'stabil_terhadap_ukuran_data': bool(stabil),
        'recall_rata2_dengan_smote': float(sm_on['recall_mean'].mean()),
        'recall_rata2_tanpa_smote': float(sm_off['recall_mean'].mean()),
        'tabel': tabel_sensitivitas_k.to_dict(orient='records'),
    },
    'verifikasi_test': {
        'n_test': int(len(X_te)),
        'n_positif_test': int(n_pos_test),
        'margin_of_error_recall': float(moe),
        'recall_k_terpilih': float(v_pil['recall_tuned']),
        'recall_k_baseline_v2': float(v_v2['recall_tuned']),
        'recall_k_terbaik_cv': float(v_bst['recall_tuned']),
        'tabel': tabel_verifikasi_k_testset.to_dict(orient='records'),
    },
    'catatan_k20_vs_k21': catatan_k20_k21,
}

simpan_json(hasil_pemilihan_k, 'hasil_pemilihan_k')

print('')
garis('SELURUH LUARAN NOTEBOOK 02')
print('Tabel  : tabel_sweep_k, tabel_one_se_kandidat, tabel_uji_antar_k, tabel_grid_knn,')
print('         tabel_heuristik_k, tabel_sensitivitas_k, tabel_verifikasi_k_testset')
print('Gambar : knn_kurva_bias_variance, knn_metrik_vs_k, knn_one_se_rule, knn_uji_antar_k,')
print('         knn_heatmap_weights, knn_heatmap_metric, knn_sensitivitas_k,')
print('         knn_verifikasi_testset')
print('JSON   : hasil_pemilihan_k')
print(f'Lokasi : {OUTPUT_DIR}')

---

# RINGKASAN UNTUK SKRIPSI

> Paragraf di bawah ini siap disalin ke Bab 3 (Metodologi) dan Bab 4 (Hasil dan Pembahasan).
> Angka di dalam kurung siku `[...]` diisi dengan nilai yang dicetak oleh CELL 18 setelah
> notebook dijalankan.

---

### A. Untuk Bab 3 - Metodologi Penentuan Nilai k

Penentuan nilai *k* pada algoritma K-Nearest Neighbors dalam penelitian ini tidak dilakukan
secara sembarang maupun diserahkan sepenuhnya kepada proses pencarian acak, melainkan melalui
empat tahap yang dapat direproduksi. **Pertama**, ruang pencarian diperluas dari tujuh kandidat
(sebagaimana pengujian awal) menjadi 26 nilai *k* ganjil pada rentang 1 sampai 51. Pembatasan
pada bilangan ganjil dilakukan secara sengaja agar mekanisme *majority voting* pada klasifikasi
biner tidak pernah menghasilkan keadaan seri. **Kedua**, setiap nilai *k* dievaluasi
menggunakan *Stratified 5-Fold Cross Validation* pada data latih (80%) dengan mencatat skor pada
fold latih maupun fold validasi, sehingga selisih keduanya dapat dipakai sebagai indikator
kuantitatif *overfitting*. **Ketiga**, nilai *k* final ditetapkan memakai **aturan
one-standard-error** (Hastie, Tibshirani & Friedman, 2009), yaitu memilih model paling sederhana
yang skornya masih berada dalam satu *standard error* dari skor terbaik. **Keempat**, keputusan
tersebut diuji ketahanannya melalui uji signifikansi statistik antar nilai *k*, pencarian grid
dua dimensi terhadap parameter `weights` dan `metric`, analisis sensitivitas terhadap ukuran data
dan penggunaan SMOTE, serta verifikasi akhir pada data uji 20% yang tidak pernah dilibatkan dalam
proses pemilihan.

---

### B. Untuk Bab 4 - Hasil Analisis Pemilihan Nilai k

Hasil *sweep* memperlihatkan pola *trade-off* bias-variance yang sangat jelas. Pada `k = 1`
model mencapai recall data latih sebesar `[train_recall k=1]` sementara recall validasinya hanya
`[val_recall k=1]`, menghasilkan selisih (*gap*) sebesar `[gap k=1]`. Selisih sebesar ini adalah
bukti langsung *overfitting*: dengan hanya satu tetangga, batas keputusan model mengikuti setiap
titik data latih sehingga menjadi bergerigi dan sangat sensitif terhadap *noise* pada pengukuran
HbA1c maupun kadar glukosa. Seiring bertambahnya *k*, selisih tersebut menyusut secara konsisten
hingga tinggal `[gap k=51]` pada `k = 51`, sesuai teori yang menyatakan komponen *variance*
pada KNN berbanding terbalik terhadap *k*. Sebaliknya, recall validasi mencapai puncak pada
`k = [K_TERBAIK_CV]` dan setelah itu mendatar - menandakan komponen *bias* mulai mendominasi
karena batas keputusan menjadi terlalu halus dan struktur lokal data hilang.

Skor validasi silang terbaik diperoleh pada `k = [K_TERBAIK_CV]` dengan recall
`[SKOR_TERBAIK]` dan *standard error* `[SE_TERBAIK]`, sehingga ambang toleransi satu
*standard error* berada pada `[AMBANG_1SE]`. Terdapat `[jumlah kandidat]` nilai *k* yang
skornya masih berada di atas ambang tersebut, yang berarti performanya tidak dapat dibedakan
secara statistik. Klaim ini dikonfirmasi oleh uji *paired t-test* dan *Wilcoxon signed-rank* atas
15 skor per nilai *k* (5-fold x 3 pengulangan), yang menunjukkan `[n]` dari `[total]` pasangan
nilai *k* tidak berbeda secara signifikan pada taraf 5%. Karena itu, memilih nilai *k* semata-mata
berdasarkan skor validasi tertinggi tidak memiliki dasar statistik yang kuat dan justru berisiko
mengoptimalkan *noise* pembagian fold. Berdasarkan aturan one-standard-error, dipilih
**`k = [K_TERPILIH]`**, yaitu nilai *k* terbesar yang masih berada dalam rentang toleransi.
Pada KNN, *k* terbesar berarti model paling sederhana karena derajat kebebasan efektifnya kira-kira
`n/k`, sehingga batas keputusan menjadi paling halus, *variance* paling rendah, dan model paling
stabil terhadap perubahan kecil pada data.

Pengujian grid dua dimensi menegaskan bahwa `weights = 'uniform'` dan `metric = 'euclidean'`
merupakan kombinasi yang tepat. Skema `weights = 'distance'` membobot tetangga terdekat jauh
lebih besar sehingga mengembalikan sebagian *variance* yang justru ingin diredam melalui
pemilihan *k* yang besar, terlihat dari *gap* train-validasi yang lebih lebar. Sementara itu,
karena kelima fitur (usia, BMI, hipertensi, HbA1c, dan kadar glukosa) telah distandarisasi dan
berdimensi rendah, jarak Euclidean bekerja optimal; metrik *chebyshev* justru merugikan karena
hanya mempertimbangkan satu fitur dengan selisih terbesar dan mengabaikan empat fitur lainnya,
padahal diagnosis diabetes bersifat multifaktor. Analisis sensitivitas menunjukkan letak *k*
optimal tidak bergeser berarti ketika ukuran data diubah menjadi 25%, 50%, dan 100%, yang
menandakan keputusan bersifat robust. Aturan praktis `k ~ sqrt(n)` yang lazim dikutip tidak
digunakan karena dengan `n = 76.916` data latih aturan tersebut menghasilkan `k = [k_sqrt_full]` -
nilai yang terlalu besar, tidak memberikan performa lebih baik, mengabaikan ketimpangan kelas
(hanya sekitar 8,5% kasus positif), sekaligus memperlambat proses inferensi pada aplikasi web.
Verifikasi akhir pada data uji 20% yang belum pernah dilihat model menghasilkan recall
`[recall test k terpilih]`, konsisten dengan estimasi validasi silang.

---

### C. Klarifikasi Penulisan "k = 20"

Perlu diklarifikasi bahwa nilai yang tercatat pada catatan revisi sebagai **k = 20** sesungguhnya
merujuk pada model KNN hasil `RandomizedSearchCV` pengujian sebelumnya, yang nilai sebenarnya
adalah **k = 21**. Perbedaan satu angka ini bukan sekadar salah ketik, melainkan konsekuensi
desain eksperimen: seluruh kandidat nilai *k* dalam penelitian ini dibatasi pada bilangan
**ganjil**. Pada klasifikasi biner (diabetes / tidak diabetes), nilai *k* genap seperti 20
berpotensi menghasilkan keadaan seri - misalnya 10 tetangga berlabel positif berhadapan dengan
10 tetangga berlabel negatif - sehingga keputusan akhir bergantung pada mekanisme *tie-breaking*
internal pustaka `scikit-learn` (memilih label dengan indeks terkecil). Mekanisme semacam itu
bersifat sewenang-wenang dan tidak dapat dipertanggungjawabkan dalam konteks skrining kesehatan.
Karena itu nilai *k* = 20 tidak pernah termasuk dalam ruang pencarian, baik pada pengujian awal
maupun pada pengujian ulang di notebook ini. Nilai `k = 21` tetap diuji secara eksplisit sebagai
*baseline* pembanding, dan hasilnya diperbandingkan langsung dengan nilai `k = [K_TERPILIH]`
yang direkomendasikan analisis one-standard-error, baik pada validasi silang maupun pada data uji.

---

### D. Kalimat Ringkas untuk Menjawab Penguji secara Lisan

> "Nilai *k* pada KNN tidak kami tentukan begitu saja. Kami menguji 26 nilai *k* ganjil dari 1
> sampai 51 dengan validasi silang 5-fold, mencatat skor latih dan validasi sekaligus. Kurvanya
> membuktikan *k* kecil overfit - pada *k* = 1 selisih skor latih dan validasi paling lebar -
> sementara *k* terlalu besar mulai underfit. Karena banyak nilai *k* di wilayah tengah ternyata
> tidak berbeda signifikan secara statistik, kami memakai aturan *one-standard-error*: di antara
> nilai *k* yang setara, dipilih model paling sederhana, yaitu *k* terbesar yang masih dalam
> toleransi satu *standard error*. Pilihan itu kami verifikasi lagi pada data uji 20% yang belum
> pernah dilihat model. Satu koreksi kecil, Pak/Bu: nilai hasil tuning kami sebenarnya *k* = 21,
> bukan 20 - kami sengaja hanya memakai *k* ganjil supaya voting pada dua kelas tidak pernah seri."